- try implementing semantic caching later at end using redis

- we had set limit max_tokens=300 for image caption generation means describe image in 300 tokens.

- embedding model
 - token per limit - 120,000
 - Rate limit (requests per minute) - 720

 - TPM — total tokens (input, since embeddings have no output tokens) processed within any rolling 60-second window
 - RPM — total number of API calls within any rolling 60-second window


Understand below
- Toekns per minute rate limit (TPM)
- Requests per minute rate limit (RPM)
- Rate Limit error

In Azure Portal-
1. I first created a Resource Group
2. I created Resource named Document Intelligence
3. I creates resource named Azure OpenAI

# second assisgnment is: take a multiple pdf with text,image,table
1. fetch the data from pdf
2. at lesat there should be 200 pages
3. if chunking(use the sementic chunking technique) required do chunking and then embedding
4. store it inside the vector database(use any of them 1. mongodb 2. astradb 3. opensearch 4.milvus) ## i have not discuss then you need to explore
5. create a index with all three index machnism(Flat, HNSW, IVF) ## i have not discuss then you need to explore
6. create a retriever pipeline
7. check the retriever time(which one is fastet)
8. print the accuray score of every similarity search
9. perform the reranking either using BM25 or MMR ## i have not discuss then you need to explore
10. then write a prompt template
11. generte a oputput through llm
12. render that output over the DOCx ## i have not discuss then you need to explore
as a additional tip: you can follow rag playlist from my youtube

Hey claude, above is my assignment on multi-modal rag, give me guidelines on how can we do this assignment, we will create multiple jupyter notebooks to complete this assignment

## Important RAGAs Metric: 

नक्की. हे चारही RAGAS metrics एकाच example वरून समजून घेतले तर खूप सोपे जातील.

समजा आपल्या RAG system ला प्रश्न आला:

> **Question:** “What causes sea level rise?”

आपल्याकडे expected/correct answer आहे:

> **Ground Truth:** “Sea level rise is mainly caused by thermal expansion of seawater and melting of glaciers and ice sheets.”

Retriever काही chunks retrieve करतो आणि मग LLM answer generate करतो.

---

## 1. Faithfulness — Answer खरंच Context वरून आला का?

**Faithfulness तपासतो की LLM ने answer मध्ये जे claims केले आहेत, ते retrieved context मध्ये supported आहेत का.**

समजा retrieved context:

```
```

```
Context:
"Ocean warming causes thermal expansion of seawater.
Melting glaciers also contribute to sea level rise."
```

LLM answer:

```
```

```
"Sea level rise is caused by thermal expansion
and melting glaciers."
```

हे context मध्ये आहे → **High Faithfulness ✅**

पण LLM म्हणाला:

```
```

```
"Sea level rise is caused by thermal expansion,
melting glaciers and earthquakes."
```

Context मध्ये `earthquakes` बद्दल काहीच नाही.

म्हणजे LLM ने स्वतःहून unsupported information add केली → **Faithfulness कमी ❌**

### लक्षात ठेव:

> **Faithfulness = Answer मधले claims retrieved Context support करतो का?**

```
```

```
Context ───────→ Answer
   ↑              │
   └─ supported? ─┘
```

**High Faithfulness = Answer well-grounded आहे / hallucination कमी आहे.**

Ground truth answer ची यासाठी गरज नसते.

---

# 2. Answer Relevancy — Answer ने विचारलेल्या Question चे उत्तर दिले का?

इथे आपण **Question ↔ Answer** relationship बघतो.

Question:

```
```

```
"What causes sea level rise?"
```

Answer:

```
```

```
"Sea level rise is primarily caused by thermal
expansion and melting land ice."
```

Directly question चे उत्तर → **High Answer Relevancy ✅**

पण answer असा असेल:

```
```

```
"Climate change affects ecosystems, weather patterns,
agriculture and global temperatures."
```

हे climate change बद्दल correct असू शकतं, पण **आपल्या specific question चे उत्तर देत नाही.**

→ **Low Answer Relevancy ❌**

### लक्षात ठेव:

> **Answer Relevancy = Generated Answer user च्या Question ला किती directly आणि appropriately address करतो?**

```
```

```
Question ──────→ Answer
       Relevant?
```

---

# 3. Context Precision — Retrieve केलेल्या chunks मध्ये relevant chunks किती focused/वर आहेत?

आता आपण LLM answer पेक्षा **Retriever** evaluate करत आहोत.

समजा Top-5 chunks:

```
```

```
#1 Thermal expansion causes sea level rise.     ✅
#2 Glacier melting contributes to sea level.    ✅
#3 Ice-sheet loss contributes to sea level.     ✅
#4 Information about forests.                   ❌
#5 Information about rainfall.                  ❌
```

Relevant chunks वरच्या positions ला आहेत.

→ **High Context Precision ✅**

पण:

```
```

```
#1 Information about forests.                   ❌
#2 Information about rainfall.                  ❌
#3 Agriculture and climate.                     ❌
#4 Thermal expansion causes sea level rise.     ✅
#5 Glacier melting contributes to sea level.    ✅
```

Correct information retrieve झाली आहे, **पण irrelevant chunks आधी आले आहेत.**

→ Context Precision कमी.

### लक्षात ठेव:

> **Context Precision = Retrieved results मध्ये relevant chunks किती focused आहेत आणि ranking मध्ये relevant chunks किती वर आहेत?**

सोप्या भाषेत:

> **“मी जे retrieve केलं त्यात किती useful/relevant आहे?”**

---

# 4. Context Recall — Correct Answer बनवण्यासाठी लागणारी पूर्ण information retrieve झाली का?

हा सुद्धा **Retriever** evaluate करतो, पण वेगळ्या angle ने.

Ground Truth मध्ये तीन important facts आहेत:

```
```

```
Correct answer requires:

1. Thermal expansion       ✅
2. Glacier melting         ✅
3. Ice-sheet loss          ✅
```

Retriever ने context मध्ये तिन्ही retrieve केले:

```
```

```
Retrieved Context:

Thermal expansion       ✅
Glacier melting         ✅
Ice-sheet loss          ✅
```

→ **High Context Recall ✅**

पण retrieve झालं:

```
```

```
Thermal expansion       ✅
Glacier melting         ❌
Ice-sheet loss          ❌
```

Correct answer तयार करण्यासाठी लागणारी बरीच information missing आहे.

→ **Low Context Recall ❌**

### लक्षात ठेव:

> **Context Recall = Correct answer तयार करण्यासाठी आवश्यक information पैकी किती information retrieved context मध्ये आली?**

सोप्या भाषेत:

> **“मला जे retrieve करायला हवं होतं, त्यापैकी किती retrieve झालं?”**

---

# Precision vs Recall — सर्वात important difference

हे interview मध्ये खूप useful आहे.

### Context Precision

> **जे retrieve केलं, त्यात किती relevant आहे?**

```
```

```
Retrieved:
✅ ✅ ✅ ❌ ❌

→ Precision चा focus
```

### Context Recall

> **जे relevant information retrieve व्हायला हवं होतं, त्यापैकी किती मिळालं?**

```
```

```
Needed:
A B C D E

Retrieved:
A B C

Missing:
D E

→ Recall चा focus
```

म्हणून:

> **Precision = Retrieved information ची QUALITY/FOCUS**
>
> **Recall = Required information ची COVERAGE/COMPLETENESS**

---

## चार metrics एकाच diagram मध्ये

```
```

```
                  QUESTION
                  /      \
                 /        \
                ↓          ↓
          RETRIEVER       ANSWER
              ↓             ↑
           CONTEXT ─────────┘
              ↑
        GROUND TRUTH


Answer Relevancy
Question ↔ Answer
"Answer question ला address करतो का?"


Faithfulness
Context ↔ Answer
"Answer context वरून supported आहे का?"


Context Precision
Retrieved Context
"Retrieved results relevant/focused आहेत का?"


Context Recall
Ground Truth ↔ Context
"Correct answer साठी लागणारी information
context मध्ये sufficiently retrieve झाली का?"
```

### Interview साठी 4 one-liners

**Faithfulness:**

> Answer मधले claims **retrieved context ने support केले आहेत का?**

**Answer Relevancy:**

> Generated answer **user च्या question ला किती relevant/direct आहे?**

**Context Precision:**

> Retrieved chunks मध्ये **relevant chunks किती focused आणि high-ranked आहेत?**

**Context Recall:**

> Correct answer तयार करण्यासाठी लागणाऱ्या information पैकी **किती information retrieve झाली?**

सगळ्यात सोपी memory trick:

> **Faithfulness → Answer ↔ Context**
>
> **Answer Relevancy → Answer ↔ Question**
>
> **Context Precision → Retrieved मध्ये किती useful?**
>
> **Context Recall → Required पैकी किती retrieved?**

### Semantic Caching

**Semantic Caching** is a technique used to **store previously generated embeddings or responses** so they can be reused instead of calling the embedding model again for repeated or semantically similar inputs.

```text
User Query / Text
       ↓
Check Cache
       ↓
   ┌── Found ──→ Reuse Cached Embedding
   │
Not Found
   ↓
Embedding Model/API
   ↓
Generate Embedding
   ↓
Store in Cache
   ↓
Use Embedding
```

**Benefits:**

* Reduces unnecessary embedding API calls
* Reduces **RPM/TPM usage**
* Reduces API cost
* Improves response time/latency
* Useful for repetitive queries or frequently processed text

**Example:** If `"What is climate sensitivity?"` has already been embedded, its stored embedding can be retrieved from a cache such as **Redis** instead of generating the embedding again.

> **Note:** Exact-text caching reuses embeddings only when the same text appears again. More advanced semantic caching can also identify **semantically similar inputs** and reuse an appropriate cached result.


# Stage 1 — Extraction Notebook Explanation

## 1. Imports & config (cells 1–3)

Loads `fitz` (PyMuPDF, for rendering PDF pages), `pandas` (for previewing results as tables), `python-dotenv` (to read `.env`), and the Azure Document Intelligence SDK.

Then it reads your Azure endpoint/key from `.env`, and sets up four folders/settings under `data/`:

- `di_cache/` — raw Azure responses, saved so you never pay to re-analyze the same PDF twice
- `images/` — cropped figure PNGs
- `extracted/` — the final output files this notebook produces
- `RENDER_DPI = 150` — the resolution used later when cropping figures out of a page

`PDF_PATHS` just globs `data/*.pdf`, so it'll pick up all three IPCC chapters automatically.

---

## 2. `get_di_client()` (cell 4)

A one-line factory that builds an authenticated `DocumentIntelligenceClient`.

Trivial, but keeping it as a function means every other cell doesn't need to know how auth works.

---

## 3. `analyze_pdf_cached()` — the core API call (cells 5–8)

This sends a PDF to Azure's `prebuilt-layout` model, which is the one model that returns text, tables, and figure locations together. The plain `prebuilt-read` model would only give you text.

The function first checks:

`data/di_cache/<pdf-name>.json`

If that file already exists, it just loads it and returns, skipping the API call entirely.

Only on a cache miss does it actually call Azure, wait for the result, and save it.

This is the single most important design choice in the notebook: since DI is billed per page and you'll be re-running cells constantly while debugging, every call after the first is free and instant.

### Cells 7–8: deliberate small test

It runs this function on just:

`PDF_PATHS[0]`

and prints how many pages, paragraphs, tables, and figures came back.

You're meant to run this before looping over all three PDFs, so if something's misconfigured — bad key, wrong endpoint, or Azure returning zero figures because of an API version issue — you find out on one document instead of after paying for all three.

---

## 4. `extract_text_units()` (cells 9–10)

Loops over every paragraph DI found and turns each into a dictionary containing:

- which document it came from
- which page it belongs to
- its role, such as body text, `pageHeader`, `pageFooter`, or `pageNumber`
- the text content itself

Headers and footers are flagged as:

`is_boilerplate: True`

but not deleted.

That filtering decision is deferred to the next notebook — **Stage 2: Cleaning** — so this notebook stays a lossless raw extraction that you can always fall back to.

---

## 5. `table_to_markdown()` + `extract_table_units()` (cells 11–13)

DI gives tables as a flat list of cells, where each cell contains a row and column index.

This logic:

1. Rebuilds those cells into a 2D grid.
2. Figures out which row is the header by checking which cells DI tagged with:
   `kind: "columnHeader"`
3. Serializes the table into a Markdown table string.

For example:

```text
| Scenario | 2050 | 2100 |
| --- | --- | --- |
| SSP1-2.6 | ... | ... |

# Stage 2 — Clean & Organize Notebook Explanation

Unchanged from what I wrote. Walking through it cell by cell:

## 1. Imports & config (cells 2-4)

Standard imports plus two tunable constants:

- `MIN_IMAGE_SIDE_PX = 80` (pixels)
- `MIN_TABLE_CELLS = 4`

These are the thresholds used later to decide what counts as "too small to be real content."

`CLEANED_DIR` (`data/cleaned/`) is created if it doesn't exist — this is where this notebook's final output goes, kept separate from Stage 1's `data/extracted/` so the raw extraction stays untouched as a fallback.

---

## 2. Load the raw streams (cell 5)

`load_jsonl()` reads a `.jsonl` file (one JSON object per line) into a plain Python list of dicts.

This loads the three files Stage 1 produced:

- `text_units`
- `table_units`
- `image_units`

and prints their raw counts as a baseline before anything gets filtered.

---

## 3. Drop boilerplate (cells 6-7)

Stage 1 tagged every text paragraph with `is_boilerplate` (`True` for headers/footers/page numbers) but didn't remove them.

This cell does two things at once: before filtering, it grabs up to 10 examples of what's about to be dropped (so you can visually confirm it's actually junk like `"IPCC AR6 WGI"` running headers, not real content), then reassigns `text_units` to only the non-boilerplate ones and prints how many were dropped.

---

## 4. De-hyphenate + normalize whitespace (cells 8-10)

Two regex patterns:

- `HYPHEN_BREAK_RE` matches a word character, then a hyphen, then whitespace, then another word character — this is the signature of a PDF line-wrap break like `"atmo- spheric"`.
- `WHITESPACE_RE` matches any run of whitespace, for collapsing it to a single space.

### Cell 9

Cell 9 is a dry-run check only — it searches (doesn't modify) the text for that hyphen pattern and prints up to 10 matches with 20 characters of surrounding context, so you can eyeball whether the pattern is correctly catching real line-break hyphens rather than something like a real hyphenated word or a range like `"1990-2000."`

### Cell 10

Cell 10 is where the actual cleaning happens: `clean_text()` merges the hyphen break (`\1\2` — drops both the hyphen and the whitespace between the two letters), collapses whitespace, and strips the ends.

It's applied in place to every text unit's `content`, then previews 5 random rows via pandas.

---

## 5. Filter degenerate tables (cells 11-12)

`count_nonempty_cells()` parses a table's Markdown string back apart: it takes the header row (`lines[0]`) plus every row after the separator (`lines[2:]`, skipping the `| --- | --- |` line), splits each on `|`, and counts non-empty cells.

Tables scoring below `MIN_TABLE_CELLS` (4) — usually a DI false-positive on a near-empty grid — get dropped.

---

## 6. Filter tiny image crops (cells 13-14)

`is_large_enough()` opens the already-saved PNG with `pymupdf.Pixmap` (just to read its width/height — no new library needed since `pymupdf` is already imported) and checks both dimensions against `MIN_IMAGE_SIDE_PX`.

Anything smaller than `80px` on either side — likely a logo or decorative icon DI mistook for a figure — gets dropped.

---

## 7. Schema validation (cells 15-16)

`validate_units()` checks every unit in a stream has non-null, non-empty:

- `doc_id`
- `page_number`
- `chunk_type`

It's run separately against all three cleaned streams and just prints `OK` or a list of violations — nothing is auto-fixed here, it's a safety check so a missing field surfaces now rather than as a broken citation in the final DOCX output.

---

## 8. Save cleaned streams (cells 17-18)

Same `save_jsonl()` pattern as Stage 1, writing the three cleaned lists to:

`data/cleaned/*.jsonl`

and printing final counts — this is exactly what Stage 3 (chunking) will read from next.

---

## Net effect

Raw extraction in → boilerplate-free, de-hyphenated, schema-validated text/table/image units out, with genuinely-empty tables and decorative images pruned.

Nothing is chunked or embedded yet.

- Semantic chunking's main drawback is computational cost, since it requires embedding text twice, during preprocessing i.e. embedding for segmentation ("The boundry check") and final embedding for storage and retrieval.
- The biggest advantage is precision—semantic chunking creates chunks that align closely with user intent during retrieval.
- semantic chunking should be used when accuracy is more important than speed, such as in domain-specific RAG systems for legal or medical domains.

# LLM / RAG Operational Metrics

Good instinct — this is exactly the kind of operational literacy that separates "I called an API" from "I understand what I'm running." Let me map out the full landscape, grouped by what they're actually measuring.

## 1. Cost

### Token-based pricing

LLMs charge separately for:

* **Input tokens** — what you send
* **Output tokens** — what the model generates

These are usually priced differently, with output tokens typically **3–4× pricier** than input tokens.

Embeddings only have an **input-token cost** since there's no generated text.

### Image / vision tokens

A picture you send to GPT-4o isn't free; it gets converted into a token count based on resolution.

Roughly:

> Larger image → more tiles → more tokens

Your figure-captioning step is burning **image tokens**, not just text tokens.

### Prompt caching

If you send the same long prefix repeatedly, such as:

* a fixed system prompt
* repeated instructions
* a common prompt template

providers can cache and discount the repeated portion.

This becomes relevant later when your **Stage 6 prompt template** reuses the same instructions across many questions.

### Batch API discount

Async, non-interactive jobs — exactly what your bulk captioning and embedding loops are — can run around **50% cheaper** through a batch endpoint instead of live calls.

The tradeoff is that results come back later instead of immediately.

### PTU vs pay-as-you-go

**Provisioned Throughput Units (PTUs)** are a flat-rate reserved-capacity option in Azure.

Two broad pricing models are:

* **Pay-as-you-go** — metered per token
* **PTU** — reserved throughput for a fixed cost

PTU only makes sense at sustained high volume. It's irrelevant at your project's scale but good to know exists.

---

## 2. Latency — Per-request speed

### TTFT — Time To First Token

**TTFT** measures how long you wait before the response starts streaming.

It is heavily influenced by:

* Prompt length
* Queueing
* Model/provider load

rather than just generation speed.

### Inter-token latency / tokens per second

Measures how quickly tokens are generated once generation starts.

This is the **tok/s** number people usually mean when benchmarking model generation speed.

### Total request latency

A rough model is:

```text
Total Request Latency
    =
TTFT
    +
(Tokens Generated ÷ Tokens Per Second)
```

This is what you actually experience end-to-end.

### Streaming vs non-streaming

With **streaming**, the user starts seeing the response after TTFT.

With **non-streaming**, the call blocks until the entire response has been generated.

Your captioning/generation calls right now aren't streamed.

---

## 3. Throughput — System-level performance

Throughput is different from individual request latency.

### Requests Per Minute — RPM

How many API requests your deployment can accept per minute.

### Tokens Per Minute — TPM

How many total tokens your deployment can process per minute.

Azure OpenAI deployments have quota limits on both:

* **RPM**
* **TPM**

If you exceed them, you can receive:

```text
HTTP 429 — Rate Limited
```

This is why production applications need retry-with-backoff logic.

### Concurrency

Concurrency measures how many requests can be in flight at the same time before the system begins:

* Queueing
* Slowing down
* Throttling requests

This is the throughput axis you'll actually exercise directly.

Your:

* bulk image-captioning loop
* bulk embedding calls

are both throughput tests against deployment **TPM/RPM limits**, even at relatively small project scale.

---

## 4. Retrieval-side metrics

This is where your assignment already points.

### QPS — Queries Per Second

Measures how many vector-search queries an index can serve per second.

### Latency percentiles

Instead of measuring only average latency, production systems often report:

* **p50**
* **p95**
* **p99**

A mean latency can hide terrible tail latency.

This is the correct way to compare your:

* FLAT
* HNSW
* IVF_FLAT

indexes in Stage 5 rather than reporting only "average time."

### Recall@k / Precision@k

These evaluate the quality of retrieved results.

**Recall@k** asks:

> Of all the relevant results that should have been retrieved, how many appeared in the top `k`?

**Precision@k** asks:

> Of the top `k` results returned, how many were actually relevant?

This requires a small ground-truth query set, as planned in Stage 5.

### MRR — Mean Reciprocal Rank

MRR rewards systems that place the **first relevant result higher** in the ranking.

For example:

```text
Relevant result at rank 1 → reciprocal rank = 1/1 = 1.0

Relevant result at rank 2 → reciprocal rank = 1/2 = 0.5

Relevant result at rank 5 → reciprocal rank = 1/5 = 0.2
```

This is especially useful when there's usually one clearly-best chunk for each question.

### Index build time vs query time

There is also a tradeoff between:

* Index creation time
* Storage/memory requirements
* Query latency
* Retrieval accuracy

For example:

```text
FLAT
  → little indexing overhead
  → exact search
  → slower query at scale

HNSW / IVF
  → additional index-building work
  → approximate search
  → faster query at scale
```

It's worth reporting **index build time and query time**, not just query latency.

---

## 5. RAG-specific answer-quality metrics

Once you start generating actual answers in **Stage 6**, another family of metrics becomes important.

Frameworks such as **RAGAS** evaluate metrics like:

### Faithfulness / groundedness

Does the generated answer actually follow from the retrieved context?

Or is the LLM hallucinating information beyond the retrieved evidence?

### Answer relevance

Does the final generated answer actually address the user's question?

### Context precision / recall

Evaluates whether the retrieved context was useful.

For example:

> Of the chunks retrieved, how many were actually relevant or needed for answering the question?

---

## 6. Operational reliability concepts

These are important production concepts, even if you don't test all of them directly at your current project scale.

### 429 handling / exponential backoff retry

The standard pattern for surviving rate limits gracefully instead of letting the application crash.

Conceptually:

```text
Request
   ↓
429 Rate Limit
   ↓
Wait
   ↓
Retry
   ↓
If another 429 occurs
   ↓
Wait longer
   ↓
Retry again
```

### Content filtering

Azure OpenAI runs content-safety checks on requests and responses.

This can:

* add a small amount of latency
* block certain requests
* block certain generated responses

It's useful to know this exists if a vision-captioning or generation call unexpectedly fails.

### Idempotency / caching

You're already implementing this with:

* DI cache
* Caption cache

The same principle is used in production systems to avoid:

* Redundant API calls
* Duplicate processing
* Unnecessary latency
* Repeated billed operations

---

# What You'll Actually Measure in This Project

## Token usage & cost

Every OpenAI / Azure OpenAI response provides usage information such as:

```text
Prompt Tokens
Completion Tokens
Total Tokens
```

You can log these per call and aggregate them to estimate total API cost.

## Latency

Wrap API and database calls with a timer.

You will already do this explicitly in **Stage 5** when benchmarking:

* FLAT
* HNSW
* IVF_FLAT

## Retrieval accuracy

You can measure:

* `Recall@k`
* `Precision@k`
* Similarity scores
* Potentially `MRR`

using your ground-truth evaluation queries.

## Throughput

Your bulk captioning and embedding loops naturally exercise throughput.

A simple useful metric is:

```text
Throughput = Number of items processed / Total elapsed time
```

For example:

```text
250 chunks / 20 seconds
= 12.5 chunks/sec
```

So a useful reported metric would be:

> **Chunks processed per second**


# Stage 3 — Current 24-Cell Notebook Walkthrough

Walking through the current, confirmed-correct version (24 cells) top to bottom:

## 1. Title (cell 1, markdown)

States the 6-step goal and the 5 required `.env` vars.

---

## 2. Imports (cell 2)

Standard library imports:

* `base64`
* `hashlib`
* `time`
* `uuid`
* etc.

Other libraries:

* `numpy`
* `pandas`
* `tiktoken` — token estimation
* `AzureOpenAI` — captioning client
* `tqdm` — progress bars
* `AzureOpenAIEmbeddings` from `langchain_openai` — embeddings client

---

## 3. Config + load data (cell 3)

Sets up paths:

* `data/cleaned`
* `data/caption_cache`
* `data/chunks`
* `data/logs`

Defines:

```python
usage_log = []
```

and:

```python
record_usage()
```

This is the shared logging mechanism every API call feeds into.

Loads the three cleaned JSONL streams from Stage 2.

---

## 4. Captioning intro (cell 4, markdown)

Introduces the image-captioning stage.

---

## 5. Captioning setup (cell 5)

Reads the 5 Azure environment variables and defines:

* `CAPTION_PROMPT`
* `get_openai_client()`
* `caption_image_cached()`

### `caption_image_cached()`

On a **cache hit**:

* Returns immediately
* Makes no API call
* Adds nothing to `usage_log`

On a **cache miss**:

1. Times the GPT-4o vision call
2. Sends the image for captioning
3. Logs real `prompt_tokens` and `completion_tokens` from `response.usage`
4. Writes the generated caption to the cache
5. Returns the caption

---

## 6–7. Sanity check (cells 6–7)

Captions only:

```python
image_units[0]
```

and prints the result.

This lets you:

* Verify caption quality
* Catch endpoint issues
* Catch deployment-name issues
* Confirm authentication works

before running captioning over every image.

---

## 8. Full captioning loop (cell 8)

Runs a `tqdm`-wrapped loop over every image.

Each image's existing:

```python
content
```

field is overwritten in place with its generated caption.

---

## 9. Embedding/chunking intro (cell 9, markdown)

Explains that embeddings now come from Azure OpenAI and why semantic chunking is implemented directly rather than using the sunset `langchain-experimental` implementation.

---

## 10. Embeddings + chunking functions (cell 10)

### `embeddings = AzureOpenAIEmbeddings(...)`

Uses the same Azure OpenAI credentials as captioning, but a different deployment:

```text
AZURE_OPENAI_EMBEDDING_DEPLOYMENT
```

### `_token_encoder`

A `tiktoken` encoder used only to estimate tokens sent to the embedding model, because LangChain's embedding wrapper does not expose the actual API usage object.

### `embed_with_logging(texts)`

This function:

1. Starts a timer
2. Estimates token usage
3. Calls the Azure OpenAI embedding deployment
4. Records usage through `record_usage()`
5. Returns the embedding vectors

Every embedding call in the notebook goes through this function.

### `split_sentences()`

Regex-based sentence splitter.

### `cosine_distance()`

Uses NumPy to calculate cosine distance between two embedding vectors.

### `semantic_chunk_page()`

Implements the semantic chunking algorithm:

```text
Sentences
   ↓
Embed all sentences in one batched call
   ↓
Calculate cosine distance between consecutive sentences
   ↓
Calculate 95th-percentile distance threshold
   ↓
Identify unusually large semantic jumps
   ↓
Split at those breakpoints
   ↓
Semantic chunks
```

---

## 11. Per-page chunking intro (cell 11, markdown)

Explains the per-page design choice:

> Exact page citations are prioritized over allowing chunks to span page boundaries.

---

## 12. Group text by page (cell 12)

Buckets `text_units` into pages using:

```python
(doc_id, page_number)
```

as the key.

Text stays in its existing reading order.

---

## 13. Run the chunker (cell 13)

For each page:

1. Join its paragraphs into one text block
2. Check the block length
3. If the page contains fewer than `300` characters, skip semantic chunking
4. Otherwise call:

```python
semantic_chunk_page()
```

The resulting chunks are stored in:

```python
text_chunks
```

---

## 14. Table/image intro (cell 14, markdown)

Introduces how tables and images are converted into the common chunk schema.

---

## 15. Reshape tables + images (cell 15)

Uses two list comprehensions to convert:

* `table_units`
* `image_units`

into the same basic schema used by text chunks.

Images additionally preserve:

```python
image_path
```

---

## 16. Unify intro (cell 16, markdown)

Introduces merging all three modalities into one shared chunk collection.

---

## 17. Merge + assign IDs (cell 17)

Combines:

```python
text_chunks
table_chunks
image_chunks
```

into:

```python
all_chunks
```

Each chunk receives:

```python
uuid4().hex
```

as its:

```python
chunk_id
```

The cell also prints the number of chunks per type.

---

## 18. Embed-every-chunk intro (cell 18, markdown)

Introduces the final embedding pass across all chunk types.

---

## 19. Embed all chunks (cell 19)

Makes one bulk call:

```python
embed_with_logging(...)
```

over every chunk's `content`.

The resulting vectors are zipped back into the chunk objects as:

```python
"embedding"
```

The cell then prints:

```python
EMBEDDING_DIM
```

This dimension is required when defining the Milvus vector-field schema in Stage 4.

---

## 20. Save chunks (cell 20)

Uses:

```python
save_jsonl()
```

to write:

```text
data/chunks/chunks.jsonl
```

This becomes the direct input to Stage 4.

---

## 21. Preview (cell 21)

Samples 2 chunks from each modality for a final spot-check:

* Text
* Table
* Image

---

## 22. Usage-summary intro (cell 22, markdown)

Introduces the API usage report and includes the caveat that the pricing values must be verified against current Azure OpenAI pricing.

---

## 23. Usage summary (cell 23)

Defines the placeholder:

```python
PRICE_PER_1K_TOKENS
```

table.

Then saves:

```python
usage_log
```

to:

```text
data/logs/stage3_usage_log.jsonl
```

If API calls were logged, the cell prints a per-`call_type` summary containing:

* Number of calls
* Tokens
* Total latency
* Average latency
* Items/sec throughput
* Tokens/sec throughput
* Estimated cost

---

## 24. Next-steps pointer (cell 24, markdown)

Hands the pipeline off to:

> **Stage 4 — Milvus + three indexes**

---

# End-to-End Result

A complete Stage 3 run performs:

```text
Cleaned Stage 2 data
        ↓
GPT-4o image captioning
        ↓
Captioned image units
        ↓
Per-page semantic text chunking
        ↓
Text + tables + images
        ↓
Shared chunk schema
        ↓
Assign chunk_id
        ↓
Azure OpenAI embeddings
        ↓
data/chunks/chunks.jsonl
        +
data/logs/stage3_usage_log.jsonl
```

So the final result is:

> **Captioned images → semantically chunked text → all three streams merged → embedded with Azure OpenAI → saved to `chunks.jsonl` → cost, latency, and throughput statistics recorded for the run.**

## Practical note for re-running

If you make changes through this chat, just close and reopen the tab before running the notebook again, to avoid the stale-cache issue we just hit.


# Token per Minute (TPM)

**Token per Minute (TPM):** a rate limit on how many tokens a specific model deployment is allowed to process within any rolling 60-second window. It's the throughput ceiling Azure enforces in real time — send more tokens than your TPM allowance within that window, and the next request gets rejected with a `429`, regardless of how few *requests* you made (that's what RPM tracks separately).

The key nuance you asked about — **what actually counts as a "token" differs by model type:**

## Generative models (e.g. GPT-4o)

Both directions count toward TPM:

* **Input tokens** — everything you send: the system prompt, your message, any retrieved context, and — for a vision-capable model like GPT-4o — an **image also gets converted into a token count** based on its resolution (that's why your captioning calls had unusually large input-token counts, not just from the text prompt)
* **Output tokens** — the tokens the model *generates* as its response (your caption text)

So one captioning call's TPM cost = input tokens (prompt + image) + output tokens (the generated caption).

This is why Foundry's Monitor page for `gpt-4o` shows non-zero numbers in **both** the Input and Output tiles.

## Embedding models (e.g. `text-embedding-3-small`)

Only one direction counts:

* **Input tokens only** — the text you're converting into a vector
* **There is no output-token concept at all** — the model's output is a fixed-length array of numbers (the vector), not generated language, so nothing gets tokenized/counted on the output side

This is exactly why you saw **"Completion token count: 0"** in the embedding deployment's metrics earlier — it's not a bug or missing data, embeddings structurally never have completion tokens.

Their entire TPM cost is just:

> How much text (in tokens) you pushed in, across however many strings were batched into that request.

## Practical takeaway for your 429 specifically

Since embeddings only ever consume *input* TPM, the only lever you have to reduce token pressure is reducing how much text you send per unit time — there's no "output token" side to optimize away, unlike with GPT-4o where a shorter `max_tokens` response would also help.


# Requests Per Minute (RPM)

**RPM (Requests Per Minute):** a rate limit on how many API *calls* a deployment is allowed to receive within any rolling 60-second window — counting each request as one unit, completely regardless of how many tokens that request contains.

## TPM vs RPM — two independent caps, both enforced simultaneously

* **TPM** cares about *volume of content* (how many tokens total)
* **RPM** cares about *volume of calls* (how many separate requests total)

You can trip either one on its own.

A concrete example from your own numbers: your `text-embedding-3-small` deployment has **RPM = 720** and **TPM = 120,000**.

If you sent 700 tiny requests (say, 10 tokens each) in a minute, you'd hit the RPM wall (700 is close to 720) while barely using any TPM budget:

```text
700 requests × 10 tokens = 7,000 tokens
```

So you would be close to the:

```text
RPM limit = 720
```

while using only:

```text
7,000 / 120,000 TPM
```

Conversely — which is exactly what happened to you — if you send far fewer requests but each one is large (your `chunk_size=200` batches, ~1600 tokens each), you can blow past the TPM ceiling while staying comfortably under the RPM one.

That's precisely what your metrics showed:

```text
~300 requests     → well under 720 RPM

~480,000 tokens   → way over 120,000 TPM
```

So in your case:

> **TPM was the bottleneck, not RPM.**

## Where the number comes from

Like TPM, RPM scales directly with the deployment's **Capacity** setting.

In your screenshot:

```text
Capacity = 120
Rate limit (RPM) = 720
```

This reflects a fixed ratio Azure applies for this model/tier — roughly:

```text
6 RPM per capacity unit
```

You don't set RPM independently.

Raising **Capacity** to fix your TPM problem will also raise your RPM ceiling proportionally as a side effect, though RPM was never your bottleneck here.


# How to Handle Rate Limit Errors for LLMs and Embedding Models

To handle rate limit errors (`HTTP 429`) for LLMs and embedding models, you must address both **Requests Per Minute (RPM)** and **Tokens Per Minute (TPM)**.

Embedding workflows are especially prone to TPM throttling because they often process large amounts of text in batches.

---

## 1. Application-Level Fault Tolerance

If a rate limit occurs, your code must handle it gracefully without crashing or entering an infinite retry loop.

### Exponential Backoff with Jitter

Do not retry immediately or at fixed intervals.

Instead, increase the wait time exponentially between attempts:

```text
1s → 2s → 4s → 8s → 16s
```

Also add a small random delay called **jitter**.

For example:

```text
Retry delay = exponential_backoff + random_jitter
```

Jitter helps prevent the **thundering herd problem**, where many failed requests retry at exactly the same time and overload the API again.

### Client-Side Concurrency Throttling

Restrict how many API calls your application can send simultaneously.

Common approaches include:

* Semaphores
* Worker limits
* Connection pools
* Async concurrency limits

Instead of allowing hundreds of embedding requests to run at once, you might allow only a controlled number of concurrent requests.

### Framework Rate Limiters

Frameworks such as **LangChain** provide rate-limiting utilities such as:

```python
InMemoryRateLimiter
```

These can deliberately pace requests before they reach the API provider.

---

## 2. Request Optimization — Batching & Chunking

How requests are structured directly affects how quickly you consume RPM and TPM.

### Smart Request Batching

Most embedding APIs allow multiple strings to be embedded in a single request:

```python
texts = [
    "chunk 1",
    "chunk 2",
    "chunk 3"
]
```

instead of:

```text
Request 1 → chunk 1
Request 2 → chunk 2
Request 3 → chunk 3
```

Batching reduces **RPM consumption** because many chunks are processed within one API request.

However, very large batches can consume too many tokens at once and cause **TPM throttling**.

Therefore, the goal is:

> Batch enough to reduce RPM, but keep each batch small enough to stay safely within TPM and per-request token limits.

### Chunk Processing with Delays

For large document-ingestion pipelines, process chunks in controlled batches and intentionally pause between batches.

For example:

```text
Batch 1
   ↓
API call
   ↓
Wait
   ↓
Batch 2
   ↓
API call
   ↓
Wait
```

You can measure each request with:

```python
start = time.time()

# API call

elapsed = time.time() - start
```

and use:

```python
time.sleep(...)
```

to control the overall request rate.

---

## 3. Architecture & Gateway Patterns

For larger production systems, putting all retry and throttling logic directly inside application code can become difficult to maintain.

Instead, rate-control responsibilities can be moved into an intermediate layer.

| Strategy                         | Description                                                                                                     | Best For                                              |
| -------------------------------- | --------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------- |
| **Asynchronous Message Queues**  | Push embedding tasks into queues such as Celery, RabbitMQ, or Redis. Workers process jobs at a controlled rate. | Bulk ingestion and offline vector indexing            |
| **API Gateways / LLM Proxies**   | Use an intermediary layer to handle retries, routing, load balancing, and throttling.                           | Production multi-user applications                    |
| **Semantic / Embedding Caching** | Store previously generated embeddings and reuse them when identical content appears again.                      | Repetitive documents, queries, or ingestion pipelines |

### Asynchronous Message Queues

Instead of sending every embedding request immediately:

```text
Documents
   ↓
Task Queue
   ↓
Rate-controlled workers
   ↓
Embedding API
   ↓
Vector Database
```

Workers consume jobs at a rate that stays inside the provider's limits.

This is particularly useful for:

* Large PDF ingestion
* Offline indexing
* Thousands or millions of chunks

### API Gateways & LLM Proxies

Tools such as:

* Portkey
* LiteLLM
* Other LLM gateway/proxy layers

can centralize:

* Retry logic
* Rate limiting
* Load balancing
* Provider fallback
* Logging
* Usage tracking

### Semantic / Embedding Caching

If a piece of text has already been embedded, avoid generating its vector again.

Conceptually:

```text
Text
  ↓
Hash text
  ↓
Check cache
  ↓
Cache hit? ── Yes → Return stored embedding
  │
  No
  ↓
Embedding API
  ↓
Store vector in cache
```

This reduces:

* API calls
* Token consumption
* Cost
* Latency
* Rate-limit pressure

---

## 4. Infrastructure & Provider Alternatives

If request optimization and retry logic are still insufficient, infrastructure-level changes may be necessary.

### Multi-Deployment / Provider Fallbacks

Production systems may route traffic to another available deployment or provider when the primary deployment is throttled.

For example:

```text
Primary embedding deployment
        ↓
      429
        ↓
Fallback deployment/provider
```

This can improve availability, although embedding-model changes must be handled carefully because vectors produced by different embedding models generally should not be mixed inside the same vector index.

### Self-Hosting Open-Weight Embedding Models

Another option is to run embedding models yourself.

Examples include:

* BGE
* E5
* Nomic embeddings
* Sentence Transformers

They can be hosted using infrastructure such as:

* Ollama
* vLLM
* Hugging Face
* Your own GPU/CPU server

This removes third-party API TPM/RPM limits, although your own compute resources then become the throughput constraint.

### Increase Provider Quota / Capacity

If the workload genuinely requires more throughput, increase the quota or deployment capacity available through the provider.

For Azure OpenAI, increasing deployment capacity can increase the available:

* TPM
* RPM

This is appropriate when throttling reflects legitimate sustained workload rather than inefficient request design.

---

# TPM vs RPM: Which Fix Helps Which Problem?

| Technique                           |              Helps RPM |               Helps TPM |
| ----------------------------------- | ---------------------: | ----------------------: |
| Batch multiple texts in one request |             ✅ Strongly |       ❌ Not necessarily |
| Reduce text/chunk size              |           ⚠️ Sometimes |                       ✅ |
| Reduce batch size                   |     ❌ May increase RPM |                       ✅ |
| Add delays between requests         |                      ✅ |                       ✅ |
| Limit concurrency                   |                      ✅ |                       ✅ |
| Exponential backoff                 |             ✅ Recovery |              ✅ Recovery |
| Cache embeddings                    |                      ✅ |                       ✅ |
| Increase deployment capacity        |                      ✅ |                       ✅ |
| Shorter LLM output                  |                      ❌ | ✅ for generative models |
| Self-host embedding model           | Removes provider limit |  Removes provider limit |

---

# Important Difference for Embedding Models

For generative LLMs:

```text
TPM consumption
=
Input Tokens
+
Output Tokens
```

For embedding models:

```text
TPM consumption
=
Input Tokens Only
```

There are no completion tokens for embeddings.

Therefore, if your embedding deployment is hitting TPM limits, the main controls are:

* Reduce the amount of text sent per unit time
* Reduce batch size
* Pace requests
* Limit concurrency
* Cache previously generated embeddings
* Increase deployment capacity

---

# Recommended Pattern for Your Embedding Pipeline

For a bulk RAG ingestion pipeline, a good architecture is:

```text
Chunks
   ↓
Token-aware batching
   ↓
Embedding cache check
   ↓
Concurrency limiter
   ↓
Embedding API
   ↓
429?
 ├── No → Save embedding
 │
 └── Yes
      ↓
 Retry-After / exponential backoff + jitter
      ↓
 Retry
```

The most practical combination is:

> **Token-aware batching + controlled concurrency + exponential backoff with jitter + caching + usage logging.**

That gives you both **rate-limit prevention** and **fault tolerance** when a `429` still occurs.


### Exact-Match Caching

**Exact-match caching:** a cache hit only happens when the new input is *identical* — byte-for-byte or character-for-character the same — to something already cached. Not "similar," not "close enough" — literally the same.

## How It Works Mechanically

1. Compute a deterministic key from the input — usually a hash (like `md5`) of the exact bytes/text, since raw text/images make awkward dictionary keys directly
2. Check if that exact key already has a stored result
3. **Hit:** return the stored result immediately, no computation/API call
4. **Miss:** do the real work (call the API, run the model), then store the result under that key for next time

```text
Input
  ↓
Generate deterministic key / hash
  ↓
Check cache
  ↓
 ┌───────────────┐
 │               │
Hit             Miss
 │               │
 ↓               ↓
Return        Call API /
cached        run model
result           ↓
 │           Store result
 │               ↓
 └──────────→ Return result
```

## You Already Have Two Working Examples in This Notebook

* `analyze_pdf_cached()` — key = the PDF's filename; if you've already analyzed `IPCC_AR6_WGI_Chapter07.pdf`, re-running never re-calls Document Intelligence for it, because it's the exact same file
* `caption_image_cached()` — key = md5 hash of the image path; the exact same cropped PNG always returns the exact same cached caption

## Why It's Helpful

* **Simple** — just a hash + a dictionary/file lookup, no ML or similarity math involved
* **Zero risk of a wrong "close enough" answer** — a hit is guaranteed to be the actual, correct result for that exact input, since nothing was approximated
* **Extremely effective against your actual failure mode** — every crash-and-rerun in this notebook re-sends the *exact same* PDFs, images, and prompt text. Since the inputs are byte-identical across attempts, exact-match caching captures essentially 100% of the redundant cost from retries — which is precisely why your Stage 1/Stage 3 captioning calls didn't get re-billed every time you re-ran cells after an error, but the currently uncached embedding calls did


# Stage 4 — Milvus Indexing Notebook Walkthrough

## 1. Title (cell 1, markdown)

States the goal, and the important Milvus constraint:

> One index type per collection, hence three separate collections.

It also includes the prerequisite command:

```bash
docker compose up -d
```

---

## 2. Imports (cell 2)

Imports the core pieces from `pymilvus`:

* `connections` — connects to the Milvus server
* `utility` — admin operations such as listing or dropping collections
* `FieldSchema` — defines individual fields
* `CollectionSchema` — defines the complete collection schema
* `DataType` — specifies Milvus data types
* `Collection` — the actual collection object used for insertion and search

---

## 3. Connect (cell 3)

```python
connections.connect(...)
```

opens a connection to Milvus at:

```text
localhost:19530
```

This is the port exposed by `docker-compose.yml`.

The connection uses the alias:

```text
"default"
```

which is Milvus's implicit default connection name. Every subsequent `Collection` or `utility` call uses this connection unless explicitly told otherwise.

Then:

```python
utility.list_collections()
```

prints any collections that already exist.

This also acts as a connection smoke test: if Milvus isn't running, this cell fails immediately with a clear connection error rather than causing something confusing later in the notebook.

---

## 4. Load chunks (cell 4)

Reads:

```text
data/chunks/chunks.jsonl
```

back into:

```python
all_chunks
```

using the same structure Stage 3 produced.

Then it determines:

```python
EMBEDDING_DIM
```

from the length of the first chunk's embedding.

That value becomes the declared dimension of Milvus's vector field in the collection schema.

---

## 5. Schema intro (cell 5, markdown)

Introduces the shared Milvus collection schema.

---

## 6. Schema + record shaping (cell 6)

### `build_schema()`

Declares seven fields:

* `chunk_id` — primary key
* `embedding` — vector field, sized to `EMBEDDING_DIM`
* `doc_id`
* `page_number`
* `chunk_type`
* `content`
* `image_path`

`content` gets Milvus's maximum allowed `VARCHAR` size:

```text
65535
```

as a safety margin against long table or page chunks.

### `to_records()`

Reshapes each chunk dictionary into exactly the fields expected by the Milvus schema.

For `image_path`, it uses:

```python
.get("image_path") or ""
```

because text and table chunks don't have an image path.

Instead of storing:

```python
None
```

it stores:

```text
""
```

because Milvus `VARCHAR` fields do not handle `None` cleanly.

Finally:

```python
records = to_records(all_chunks)
```

is run once.

The resulting `records` list is reused for all three collections because the underlying data is identical — only the vector index differs.

---

## 7. Index configs intro (cell 7, markdown)

Explains:

* `FLAT`
* `HNSW`
* `IVF_FLAT`

and what their tunable parameters mean.

---

## 8. `INDEX_CONFIGS` (cell 8)

Defines a dictionary containing three named configurations.

Each configuration bundles:

* `index_type`
* `metric_type`
* `index_params`
* `search_params`

All three use:

```text
COSINE
```

similarity.

### Build-time parameters

`index_params` are used when the index is created.

### Search-time parameters

`search_params` are used whenever a query is executed.

Keeping both together means Stage 5 can simply use something like:

```python
INDEX_CONFIGS["hnsw"]["search_params"]
```

instead of manually recreating the corresponding search configuration.

---

## 9. Build-function intro (cell 9, markdown)

Introduces the collection-building function.

---

## 10. `build_collection()` (cell 10)

This is the core builder.

It performs the following steps:

```text
Check for existing collection
        ↓
Drop stale collection
        ↓
Create fresh collection
        ↓
Insert all records
        ↓
flush()
        ↓
Build vector index
        ↓
load()
```

### Drop existing collection

Any existing collection with the same name is deleted first.

This makes the notebook safe to re-run and effectively idempotent.

### Create collection

A fresh `Collection` is created using the shared schema.

### Insert records

All chunks are inserted into the collection.

### `flush()`

```python
collection.flush()
```

forces Milvus to persist the newly inserted data before indexing.

### `create_index()`

Creates the vector index on the:

```text
embedding
```

field using that collection's corresponding index configuration.

### `.load()`

Finally:

```python
collection.load()
```

loads the collection into memory.

Milvus requires a collection to be loaded before it can be searched.

---

## 11. Actual build loop (cell 11)

Loops through:

```python
INDEX_CONFIGS
```

and calls `build_collection()` for each configuration.

It creates:

```text
chunks_flat
chunks_hnsw
chunks_ivf_flat
```

The entire:

```text
Create collection
+ Insert
+ Flush
+ Build index
+ Load
```

process is timed using:

```python
time.perf_counter()
```

and the notebook prints each collection's:

* Build time
* Number of rows

as it finishes.

---

## 12. Smoke-test intro (cell 12, markdown)

Introduces a simple correctness test before doing real benchmarking.

---

## 13. Smoke-test search (cell 13)

Uses:

```python
all_chunks[0]["embedding"]
```

as a temporary query vector.

This is **not a real user query**. It is simply a convenient vector whose expected answer is already known.

For each of the three collections, the notebook runs:

```python
collection.search(...)
```

using that index's own:

* `metric_type`
* `search_params`

from `INDEX_CONFIGS`.

The search requests the top 3 results and several metadata fields.

### `hit.distance`

Returns the similarity score.

### `hit.entity.get(...)`

Retrieves the requested metadata fields from the returned chunk.

Because the query vector is literally chunk 0's own embedding:

> The top result in every collection should be chunk 0 itself with a cosine similarity close to `1.0`.

So this acts as a simple correctness check.

---

## 14. Row-count validation (cell 14)

Confirms that:

```python
collection.num_entities
```

matches:

```python
len(all_chunks)
```

for every collection.

Conceptually:

```text
Milvus rows == Number of Stage 3 chunks
```

This catches silent or partial inserts before they create confusing results during Stage 5 benchmarking.

---

## 15. Next-steps pointer (cell 15, markdown)

Hands the pipeline off to:

> **Stage 5 — Retriever pipeline, latency & accuracy benchmarking**

---

# Net Effect of a Full Run

```text
data/chunks/chunks.jsonl
          ↓
     Load chunks
          ↓
   Shared schema
          ↓
 ┌────────┼─────────┐
 ↓        ↓         ↓
FLAT     HNSW    IVF_FLAT
 ↓        ↓         ↓
Collection Collection Collection
 ↓        ↓         ↓
Insert identical chunks
 ↓        ↓         ↓
Build respective index
 ↓        ↓         ↓
Load into memory
 └────────┼─────────┘
          ↓
     Smoke test
          ↓
  Validate row counts
```

The final result is:

* `chunks_flat`
* `chunks_hnsw`
* `chunks_ivf_flat`

All three collections are:

* Fully populated
* Indexed
* Loaded into memory
* Confirmed to return sensible results
* Ready for Stage 5 benchmarking

And **zero Azure OpenAI API calls** are made anywhere in Stage 4.

---

# Starting Milvus with Docker

Yes — run:

```bash
docker compose up -d
```

in your **project root directory**, which is:

```text
/Users/rutikakadam/Documents/My Studies/Projects/multi-modal-rag
```

This is the directory where your:

```text
docker-compose.yml
```

file lives.

So in the terminal:

```bash
cd "/Users/rutikakadam/Documents/My Studies/Projects/multi-modal-rag"

docker compose up -d
```

## Before Running the Command

Make sure **Docker Desktop is already running** on your Mac.

Otherwise, you may get an error such as:

```text
Cannot connect to the Docker daemon
```

## Why `-d`?

The:

```text
-d
```

means **detached mode**.

Instead of keeping the terminal occupied with container logs:

```text
docker compose up
```

it starts the containers in the background:

```text
docker compose up -d
```

and immediately gives you the terminal prompt back.

## Check That Milvus Started Successfully

Run:

```bash
docker compose ps
```

You should see containers such as:

```text
milvus-etcd
milvus-minio
milvus-standalone
```

Their status should eventually show:

```text
Up
```

or:

```text
healthy
```

Once `milvus-standalone` is running properly, you can go back to `4_milvus_indexing.ipynb` and run the Milvus connection cell.


# Docker Compose Configuration for Local Milvus

## 1. What is a `.yml` / `.yaml` file?

**YAML** stands for **YAML Ain't Markup Language**.

Files using YAML normally have one of these extensions:

```text
.yml
.yaml
```

YAML is a **human-readable configuration file format** used to describe settings and infrastructure in a structured way.

A configuration file generally does not contain the main application logic. Instead, it tells a program **how something should be configured, started, or connected**.

For example:

```yaml
ports:
  - "19530:19530"
```

is not application logic. It is configuration telling Docker how a port should be mapped.

### Code vs Configuration

```text
Python (.py)
    ↓
Defines application/program logic

YAML (.yml)
    ↓
Defines configuration/settings
```

YAML is commonly used with technologies such as:

- Docker Compose
- Kubernetes
- GitHub Actions
- CI/CD pipelines
- Application configuration files

---

# 2. What is `docker-compose.yml`?

`docker-compose.yml` is a YAML configuration file used by **Docker Compose**.

Instead of manually starting and configuring multiple Docker containers, we describe all required services inside one file.

For this project:

```text
docker-compose.yml
        ↓
Docker Compose
        ↓
Creates and manages
        ↓
┌───────────────────────┐
│ etcd                  │
│ MinIO                 │
│ Milvus Standalone     │
└───────────────────────┘
```

Therefore, `docker-compose.yml` is essentially the **blueprint for the local Milvus infrastructure**.

It specifies:

- Which services should run
- Which Docker images should be used
- Which ports should be exposed
- Which environment variables should be provided
- Where persistent data should be stored
- How containers communicate
- Which services depend on other services
- How Docker checks whether services are healthy

---

# 3. Why does Milvus need three services?

Our Docker Compose setup contains:

```text
1. etcd
2. MinIO
3. Milvus Standalone
```

Conceptually:

```text
                    Milvus
                 Vector Database
                  /           \
                 /             \
              etcd             MinIO
        Metadata/State      Object Storage
```

### Milvus

Milvus is the actual **vector database engine**.

Our RAG pipeline stores embeddings in Milvus and performs vector similarity search.

For example:

```text
Document Chunk
      ↓
OpenAI Embedding Model
      ↓
1536-dimensional Vector
      ↓
Milvus
```

Milvus is where we create and compare indexes such as:

```text
FLAT
HNSW
IVF_FLAT
```

---

### etcd

**etcd** is a distributed key-value store.

Milvus uses etcd for **metadata, system state, and coordination information**.

A simplified mental model is:

```text
etcd
 ↓
"Information about the database"
```

rather than thinking of etcd as the main place where our embedding vectors are stored.

---

### MinIO

**MinIO** is an object-storage system compatible with the S3-style storage model.

Milvus uses object storage for persistent data/files.

Simplified:

```text
etcd
→ metadata / coordination

MinIO
→ persistent object storage

Milvus
→ vector database engine
```

---

# 4. Docker Compose Version

```yaml
version: '3.5'
```

This indicates the Docker Compose file format/version for which this configuration was originally written.

Modern Docker Compose versions generally no longer require the top-level `version` field, so Docker may show it as obsolete. Older Compose files commonly include it.

---

# 5. `services:`

```yaml
services:
```

This section defines the applications/services Docker Compose should run.

Our file defines:

```text
services
│
├── etcd
│
├── minio
│
└── standalone
```

Each service normally becomes a Docker container.

---

# 6. etcd Service

```yaml
etcd:
  container_name: milvus-etcd
  image: quay.io/coreos/etcd:v3.5.5
```

### `container_name`

```yaml
container_name: milvus-etcd
```

Specifies the name of the running Docker container.

Therefore:

```text
Service name   = etcd
Container name = milvus-etcd
```

---

### `image`

```yaml
image: quay.io/coreos/etcd:v3.5.5
```

A **Docker Image** is a packaged blueprint used to create a container.

Here Docker uses the etcd `v3.5.5` image.

```text
Docker Image
      ↓
Create/Start
      ↓
Docker Container
```

An easy analogy:

```text
Recipe              → Docker Image
Prepared dish       → Docker Container
```

The image is the definition/package; the container is the running instance.

---

# 7. Environment Variables

```yaml
environment:
  - ETCD_AUTO_COMPACTION_MODE=revision
  - ETCD_AUTO_COMPACTION_RETENTION=1000
  - ETCD_QUOTA_BACKEND_BYTES=4294967296
  - ETCD_SNAPSHOT_COUNT=50000
```

**Environment variables** provide configuration values to the application running inside a container.

These settings configure etcd behavior.

For example:

```text
ETCD_AUTO_COMPACTION_MODE
→ controls automatic cleanup/compaction behavior

ETCD_AUTO_COMPACTION_RETENTION
→ determines how much revision history should be retained

ETCD_QUOTA_BACKEND_BYTES
→ sets the backend database size quota

ETCD_SNAPSHOT_COUNT
→ controls snapshot-related behavior
```

These values are infrastructure configuration and generally do not need to be changed for this assignment.

---

# 8. Docker Volumes and Persistent Storage

For etcd:

```yaml
volumes:
  - ./volumes/etcd:/etcd
```

This maps a directory on the **host machine** to a directory inside the **Docker container**.

Syntax:

```text
HOST_PATH : CONTAINER_PATH
```

Therefore:

```text
./volumes/etcd : /etcd
       ↑            ↑
      Mac       Container
```

This is important because Docker containers are disposable/re-creatable.

Without persistent storage:

```text
Container
   ↓
Data inside container
   ↓
Container removed
   ↓
Data may be lost
```

With a volume:

```text
Container
     ↕
Volume mapping
     ↕
Mac filesystem

Container recreated
     ↓
Existing persistent data remains available
```

Our project therefore has persistent directories such as:

```text
volumes/
│
├── etcd/
├── minio/
└── milvus/
```

---

# 9. etcd Command

```yaml
command: etcd -advertise-client-urls=http://127.0.0.1:2379 \
              -listen-client-urls http://0.0.0.0:2379 \
              --data-dir /etcd
```

The `command` tells Docker what command should run when the container starts.

Important part:

```text
2379
```

is the port on which etcd accepts client communication inside the Docker environment.

Milvus later connects to:

```text
etcd:2379
```

---

# 10. Health Checks

etcd contains:

```yaml
healthcheck:
  test: ["CMD", "etcdctl", "endpoint", "health"]
  interval: 30s
  timeout: 20s
  retries: 3
```

A **health check** allows Docker to periodically verify that a service is actually functioning.

Conceptually:

```text
Container running
       ↓
Docker performs health check
       ↓
Application responding?
      / \
    Yes  No
     ↓    ↓
 Healthy Retry
```

Here Docker checks every:

```text
30 seconds
```

allows up to:

```text
20 seconds
```

for the check, and retries:

```text
3 times
```

before the service can be considered unhealthy.

---

# 11. MinIO Service

```yaml
minio:
  container_name: milvus-minio
  image: minio/minio:RELEASE.2023-03-20T20-16-18Z
```

This creates the MinIO container.

```text
Service       = minio
Container     = milvus-minio
Purpose       = Object storage
```

---

# 12. MinIO Credentials

```yaml
environment:
  MINIO_ACCESS_KEY: minioadmin
  MINIO_SECRET_KEY: minioadmin
```

These configure local MinIO credentials.

For this local development setup:

```text
Access Key = minioadmin
Secret Key = minioadmin
```

These simple/default credentials are acceptable for a local learning environment but should **not** be used as production credentials.

Production secrets should normally be supplied securely rather than hard-coded into a committed Compose file.

---

# 13. Port Mapping

MinIO has:

```yaml
ports:
  - "9001:9001"
  - "9000:9000"
```

Docker port mapping uses:

```text
HOST_PORT : CONTAINER_PORT
```

For example:

```text
9000:9000

Mac
localhost:9000
      ↓
Docker
      ↓
MinIO container
port 9000
```

For this MinIO setup:

```text
9000 → MinIO API
9001 → MinIO web console
```

---

# 14. MinIO Persistent Storage

```yaml
volumes:
  - ./volumes/minio:/minio_data
```

Meaning:

```text
Mac:
./volumes/minio

        ↕

Container:
/minio_data
```

This allows MinIO data to persist when containers are stopped/recreated.

---

# 15. MinIO Startup Command

```yaml
command: minio server /minio_data --console-address ":9001"
```

This tells MinIO to:

```text
Start MinIO server
       ↓
Use /minio_data for storage
       ↓
Run web console on port 9001
```

---

# 16. MinIO Health Check

```yaml
healthcheck:
  test: ["CMD", "curl", "-f", "http://localhost:9000/minio/health/live"]
```

Docker sends a request to MinIO's health endpoint.

If MinIO responds successfully, the container is considered healthy.

---

# 17. Milvus Standalone Service

```yaml
standalone:
  container_name: milvus-standalone
  image: milvusdb/milvus:v2.4.9
```

This is the **actual Milvus vector database**.

```text
Service       = standalone
Container     = milvus-standalone
Image         = Milvus v2.4.9
```

---

# 18. Why is it called `standalone`?

```yaml
command: ["milvus", "run", "standalone"]
```

This tells Milvus to run in **Standalone mode**.

Milvus can be deployed in different architectures.

Simplified:

```text
Milvus Standalone
→ Single-node deployment
→ Simple local setup
→ Good for development/testing/assignments

Milvus Distributed
→ Multiple distributed components
→ Designed for much larger production workloads
```

For this RAG assignment, **Milvus Standalone is sufficient**.

---

# 19. Connecting Milvus to etcd and MinIO

```yaml
environment:
  ETCD_ENDPOINTS: etcd:2379
  MINIO_ADDRESS: minio:9000
```

This tells Milvus where its supporting services are.

```text
Milvus
  │
  ├────→ etcd:2379
  │
  └────→ minio:9000
```

The important point is that:

```text
etcd
minio
```

are Docker Compose **service names**.

Docker's internal networking allows containers to communicate using these names.

Therefore Milvus does not need to know the containers' changing internal IP addresses.

---

# 20. Host `localhost` vs Docker Service Names

This is an important Docker concept.

From our Jupyter notebook running directly on the Mac:

```text
localhost
→ our Mac
```

Therefore Python connects to Milvus using:

```python
host="localhost"
port="19530"
```

But from one Docker container to another:

```text
localhost
→ that SAME container
```

Therefore Milvus should not use:

```text
localhost:2379
```

to find the separate etcd container.

Instead it uses:

```text
etcd:2379
```

because `etcd` is the service name available through Docker networking.

So:

```text
Outside Docker:

Jupyter
   ↓
localhost:19530
   ↓
Milvus


Inside Docker:

Milvus
   ├──→ etcd:2379
   └──→ minio:9000
```

---

# 21. Milvus Volume

```yaml
volumes:
  - ./volumes/milvus:/var/lib/milvus
```

Maps:

```text
Mac:
./volumes/milvus

        ↕

Container:
/var/lib/milvus
```

This provides persistent local storage for Milvus.

---

# 22. Milvus Health Check

```yaml
healthcheck:
  test: ["CMD", "curl", "-f", "http://localhost:9091/healthz"]
  interval: 30s
  start_period: 90s
  timeout: 20s
  retries: 3
```

Docker checks whether Milvus is healthy.

The additional:

```yaml
start_period: 90s
```

gives Milvus time to initialize before early health-check failures are counted normally.

This is useful because databases can take some time to start.

---

# 23. Milvus Ports

```yaml
ports:
  - "19530:19530"
  - "9091:9091"
```

The most important port for our project is:

```text
19530
```

Our Python/PyMilvus client connects through it.

For example:

```python
from pymilvus import connections

connections.connect(
    alias="default",
    host="localhost",
    port="19530"
)
```

Flow:

```text
Jupyter Notebook
       ↓
localhost:19530
       ↓
Docker port mapping
       ↓
Milvus Container
       ↓
Milvus port 19530
```

Port `9091` is used for Milvus management/health-related endpoints in this setup.

---

# 24. `depends_on`

```yaml
depends_on:
  - "etcd"
  - "minio"
```

This indicates that the Milvus `standalone` service depends on:

```text
etcd
MinIO
```

Therefore the startup relationship is approximately:

```text
etcd ────┐
         ├──→ Milvus
MinIO ───┘
```

A useful nuance:

`depends_on` primarily expresses service dependency/startup ordering. In this simple form, it does not necessarily guarantee that etcd and MinIO are fully healthy before Milvus begins starting.

The individual `healthcheck` configurations still provide service-health monitoring.

---

# 25. Docker Network

At the bottom:

```yaml
networks:
  default:
    name: milvus
```

Docker creates a network called:

```text
milvus
```

The containers communicate over this internal Docker network.

```text
Docker Network: milvus

┌─────────────────────────────────────────┐
│                                         │
│            Milvus Standalone            │
│              /         \                │
│             /           \               │
│     milvus-etcd      milvus-minio       │
│                                         │
└─────────────────────────────────────────┘
```

This provides isolation and allows service-name-based communication.

---

# 26. What happens when `docker compose up -d` is executed?

From the directory containing `docker-compose.yml`:

```bash
docker compose up -d
```

Docker Compose reads the configuration and approximately performs:

```text
Read docker-compose.yml
        ↓
Create Docker network
        ↓
Pull required Docker images
(if not already available)
        ↓
Create containers
        ↓
Attach volumes
        ↓
Configure environment variables
        ↓
Configure ports
        ↓
Start etcd
        ↓
Start MinIO
        ↓
Start Milvus
        ↓
Run health checks
```

The `-d` option means:

```text
detached mode
```

so the containers continue running in the background and the terminal is returned to us.

---

# 27. Useful Docker Commands

Start the complete Milvus environment:

```bash
docker compose up -d
```

Check running containers:

```bash
docker ps
```

Check Compose services:

```bash
docker compose ps
```

View logs:

```bash
docker compose logs
```

Follow logs continuously:

```bash
docker compose logs -f
```

Stop and remove the Compose containers/network:

```bash
docker compose down
```

Restart services:

```bash
docker compose restart
```

---

# 28. Important: `stop`, `down`, and deleting volumes are different

Stopping containers does **not** automatically mean deleting the database data.

Because we use bind-mounted directories:

```text
./volumes/etcd
./volumes/minio
./volumes/milvus
```

our persistent data exists outside the disposable container layer.

Conceptually:

```text
Container
   ↓
Can be stopped/recreated

Persistent volume directory
   ↓
Data remains
```

Therefore, be careful about manually deleting the `./volumes/` directory because that can remove the persisted local Milvus data.

---

# 29. Full Architecture of Our Local Milvus Setup

```text
                    YOUR MACBOOK
                         │
                         │
              Jupyter / Python
                         │
                    PyMilvus
                         │
                localhost:19530
                         │
                         ▼
┌──────────────── Docker Environment ────────────────┐
│                                                    │
│                 Milvus Standalone                  │
│                Vector DB Engine                    │
│                   /        \                       │
│                  /          \                      │
│                 ▼            ▼                     │
│              etcd          MinIO                   │
│          Metadata/State   Object Storage            │
│                                                    │
│             Docker Network: milvus                 │
└────────────────────────────────────────────────────┘
                         │
                         ▼
                Persistent folders
                         │
        ┌────────────────┼────────────────┐
        ▼                ▼                ▼
 volumes/etcd      volumes/minio     volumes/milvus
```

---

# 30. How This Fits into the RAG Assignment

Our complete pipeline becomes:

```text
IPCC PDFs
    ↓
Document Intelligence
    ↓
Text + Tables + Figures
    ↓
Cleaning
    ↓
Image Captioning
    ↓
Semantic Chunking
    ↓
OpenAI Embedding Model
    ↓
Embedding Vectors
    ↓
PyMilvus
    ↓
Milvus running locally in Docker
    ↓
Collections
    ↓
FLAT / HNSW / IVF_FLAT
    ↓
Vector Retrieval
    ↓
Retrieval-Time Comparison
    ↓
Retrieval Accuracy Evaluation
    ↓
Reranking
    ↓
LLM
    ↓
Final RAG Response
```

---

# 31. Local Development vs Future Production Deployment

Currently:

```text
My Mac
   ↓
Docker
   ↓
Milvus
```

This is a **local development environment**.

If the application is later deployed publicly, our laptop should not remain the production database server.

Production could instead look like:

```text
Users
  ↓
Internet
  ↓
Deployed Backend
  ↓
Cloud/Managed Milvus
```

For example, Milvus could be hosted using a managed service or deployed on cloud infrastructure.

Therefore:

> Using Docker locally does **not** mean the final application must always depend on the local laptop.

Docker is currently giving us a reproducible local Milvus development environment.

---

# 32. Key Terms to Remember

| Term | Meaning |
|---|---|
| **YAML** | Human-readable configuration file format |
| **Configuration** | Settings/instructions describing how software should be set up |
| **Docker** | Platform for running applications in containers |
| **Docker Image** | Packaged blueprint used to create a container |
| **Docker Container** | Running instance created from an image |
| **Docker Compose** | Tool for defining/managing multiple related containers |
| **Service** | Application/component defined inside Docker Compose |
| **Volume** | Mechanism for persistent data storage |
| **Port Mapping** | Connects a host-machine port to a container port |
| **Environment Variable** | Configuration value supplied to an application |
| **Health Check** | Test used to determine whether a service is functioning |
| **Docker Network** | Private network allowing containers to communicate |
| **etcd** | Metadata/state/coordination store used by Milvus |
| **MinIO** | Object-storage service used by Milvus |
| **Milvus** | Vector database used for embedding storage and similarity search |
| **Standalone** | Single-node Milvus deployment suitable for local development |

---

# Final Mental Model

Do not think of `docker-compose.yml` as Python/application code.

Think of it as:

> **A blueprint describing the infrastructure required to run Milvus locally.**

Our file tells Docker:

```text
"Create an etcd service"
        +
"Create a MinIO service"
        +
"Create a Milvus standalone service"
        +
"Connect them through a Docker network"
        +
"Expose Milvus on port 19530"
        +
"Persist their data in local folders"
        +
"Monitor them using health checks"
```

Then our Python application only needs to connect to:

```text
localhost:19530
```

and Docker Compose handles the supporting Milvus infrastructure behind the scenes.

### `docker compose up -d`

Starts all services defined in the `docker-compose.yml` file and runs them in the **background (detached mode)**.

```bash
docker compose up -d
```

* **`docker`** → invokes Docker.
* **`compose`** → uses the `docker-compose.yml` configuration.
* **`up`** → creates and starts the defined containers/services.
* **`-d`** → runs the containers in **detached/background mode**, freeing the terminal.

For our Milvus setup, it starts:

```text
docker-compose.yml
       ↓
docker compose up -d
       ↓
├── etcd
├── MinIO
└── Milvus Standalone
       ↓
All run in background
```

**In short:** `docker compose up -d` = **Start the complete Docker Compose environment in the background.**


# What Are We Actually Doing? — Latency & Accuracy Benchmarking

Stepping back from the code to the actual goal: **you're trying to answer "is it worth using an approximate (faster) index instead of an exact (slower) one, and how much accuracy would that cost me?"**

Benchmarking is just the disciplined way of answering that instead of guessing.

---

## The Core Tension Being Measured

* **FLAT** = exact search. Compares your query against *every single* stored vector, guaranteed to find the true best matches. Correct, but doesn't scale — gets slower as data grows.
* **HNSW / IVF_FLAT** = approximate search. Uses a shortcut structure (a graph, or pre-clustered groups) to avoid comparing against everything, so it's faster — but "faster by skipping work" means it *might* occasionally miss the true best match in exchange for speed.

Benchmarking exists because that tradeoff is only a hypothesis until you actually measure it on your own data.

Maybe the approximate indexes lose zero accuracy and are much faster — **great trade**.

Maybe they lose a lot of accuracy for barely any speed gain — **bad trade**.

You can't know which without measuring both sides.

---

## What "Latency Benchmarking" Means Here

Timing how long each index takes to answer the **same search request**, repeated across 20 different questions.

You don't test only once because one run could be unusually fast or slow.

For every index:

```text
Question
   ↓
Same query vector
   ↓
Milvus .search()
   ↓
Measure search time
```

You repeat this across all 20 questions for:

* FLAT
* HNSW
* IVF_FLAT

Then summarize the timings using:

### Mean

The average search latency across all queries.

```text
Mean latency
=
Sum of all query latencies
÷
Number of queries
```

### p50

The **median latency**.

50% of requests finished faster than this value, and 50% took longer.

This represents a typical request.

### p95

The latency below which **95% of requests** finished.

Only the slowest 5% took longer.

This is important because the average alone can hide occasional very slow requests.

So latency benchmarking gives you the:

> **Speed side of the index comparison.**

---

## What "Accuracy Benchmarking" Means Here

For each of the 20 questions, you already **know the correct chunk**.

Why?

Because the evaluation process works like this:

```text
Real chunk
   ↓
GPT-4o generates a question
that this chunk answers
   ↓
Question + original chunk_id
stored as ground truth
```

So if a question was generated from:

```text
chunk_id = abc123
```

then:

```text
abc123
```

is the known-correct result for that query.

Now you search each Milvus index and ask:

> Did this index return `abc123` somewhere in its top 5 results?

That is what your **Recall@5** calculation checks.

### Example

Suppose you have 20 questions.

If HNSW retrieves the known-correct chunk within the top 5 for 18 questions:

```text
Recall@5
=
18 / 20
=
0.90
=
90%
```

If FLAT retrieves it for all 20:

```text
Recall@5
=
20 / 20
=
100%
```

So Recall@5 gives you the:

> **Accuracy side of the index comparison.**

You also record the **top-1 cosine similarity score** as another retrieval signal showing how strongly the best returned result matches the query.

---

## Why You Need Both Together

Speed alone tells you nothing about whether the retrieved information is correct.

Accuracy alone tells you nothing about whether retrieval is fast enough.

You therefore compare:

```text
                    INDEX BENCHMARK
                           │
              ┌────────────┴────────────┐
              ↓                         ↓
           LATENCY                   ACCURACY
           "How fast?"               "How good?"
              ↓                         ↓
      Mean / p50 / p95              Recall@5
                               Avg top-1 similarity
              └────────────┬────────────┘
                           ↓
                  Choose best tradeoff
```

For example:

### Scenario 1 — Good Tradeoff

```text
FLAT
Latency = 10 ms
Recall@5 = 100%

IVF_FLAT
Latency = 5 ms
Recall@5 = 100%
```

IVF_FLAT is roughly **2× faster while maintaining the same retrieval accuracy**.

That is a strong tradeoff.

### Scenario 2 — Bad Tradeoff

```text
FLAT
Latency = 10 ms
Recall@5 = 100%

IVF_FLAT
Latency = 5 ms
Recall@5 = 60%
```

IVF_FLAT is faster, but it misses the correct chunk in many queries.

That speed improvement probably isn't worth the accuracy loss.

---

## What the Benchmark Is Ultimately Trying to Answer

Your Stage 5 experiment is essentially asking:

> **Which index gives me the best balance between retrieval speed and retrieval accuracy for my MM-RAG dataset?**

You are comparing:

```text
FLAT
vs.
HNSW
vs.
IVF_FLAT
```

using the **same dataset, same questions, same query embeddings, same top-K, and same cosine similarity metric**.

That makes it a fair comparison.

The benchmark's whole job is to reveal which index gives the best practical tradeoff **on your own dataset**, instead of assuming that an approximate index is faster or "almost as accurate" just because that is generally true at scale.


# Latency in RAG & Vector Search

## 1. What is Latency?

**Latency** is the amount of time a system takes to process a request and return a result.

In simple terms:

> **Latency = How long one request takes to get a response.**

It is commonly measured in:

- milliseconds (`ms`)
- seconds (`s`)

Example:

```text
Search Request
     ↓
     ↓  9.1 ms
     ↓
Search Results
```

If a vector database takes **9.1 ms** to return the Top-K results:

```text
Search Latency = 9.1 milliseconds
```

---

# 2. Latency in a RAG Pipeline

A typical RAG retrieval pipeline contains multiple steps:

```text
User Query
    ↓
Query Embedding
    ↓
Vector Database Search
    ↓
Top-K Documents
    ↓
Reranking
    ↓
Prompt Construction
    ↓
LLM Generation
    ↓
Final Answer
```

Each stage introduces its own latency.

Therefore, "latency" must always be defined in terms of **what part of the system is being measured**.

---

# 3. Different Types of Latency in RAG

For example:

```text
User Query
    │
    ├── Embedding Latency
    │
    ▼
Query Vector
    │
    ├── Vector Search Latency
    │
    ▼
Retrieved Documents
    │
    ├── Reranking Latency
    │
    ▼
Reranked Documents
    │
    ├── LLM Generation Latency
    │
    ▼
Final Answer
```

### Query Embedding Latency

Time required to convert the user's query into an embedding vector.

```text
"What causes sea level rise?"
           ↓
     Embedding Model
           ↓
[0.23, -0.17, 0.84, ...]
```

If this takes:

```text
120 ms
```

then:

```text
Embedding Latency = 120 ms
```

---

### Vector Search / Index Search Latency

Time required by the vector database to search the index and return the Top-K nearest vectors.

```text
Query Vector
     ↓
Milvus Index
     ↓
Similarity Search
     ↓
Top-K Results
```

Example:

```text
Search starts
     ↓
Milvus HNSW search
     ↓
Top-5 results returned

Time = 9.1 ms
```

Therefore:

```text
Index Search Latency = 9.1 ms
```

---

### Reranking Latency

If BM25, MMR, Cross-Encoder, etc. are used after initial retrieval:

```text
Retrieved Documents
        ↓
     Reranker
        ↓
Reordered Documents
```

the time spent here is **reranking latency**.

---

### LLM Generation Latency

Time required by the LLM to generate the final answer.

```text
Prompt
  ↓
LLM
  ↓
Generated Answer
```

This can often be much larger than vector search latency.

---

# 4. End-to-End Latency

**End-to-end latency** measures the total time from receiving the user's request until the requested final output is returned.

For a full RAG application:

```text
TIMER START
     ↓
User Query
     ↓
Embedding
     ↓
Vector Search
     ↓
Reranking
     ↓
Prompt Construction
     ↓
LLM Generation
     ↓
Final Answer
     ↓
TIMER STOP
```

Conceptually:

```text
End-to-End Latency
≈
Embedding Latency
+
Retrieval Latency
+
Reranking Latency
+
LLM Latency
+
Other application/network overhead
```

---

# 5. Retrieval Latency vs End-to-End Latency

These should not be confused.

### Index Search Latency

```text
Query Vector
    ↓
Vector DB Search
    ↓
Top-K Results
```

Measures only vector database/index performance.

### End-to-End Retrieval Latency

```text
User Query
    ↓
Embedding
    ↓
Vector DB Search
    ↓
Top-K Results
```

Includes query embedding + vector search + associated overhead.

### Full RAG End-to-End Latency

```text
User Query
    ↓
Embedding
    ↓
Retrieval
    ↓
Reranking
    ↓
LLM
    ↓
Final Answer
```

Measures the complete user-facing RAG pipeline.

---

# 6. Latency Benchmarking

**Latency benchmarking** means repeatedly measuring how long a system/component takes to process representative requests.

One measurement is usually not reliable.

For example:

```text
Query 1 → 8.2 ms
Query 2 → 9.1 ms
Query 3 → 7.8 ms
...
Query 20 → 8.7 ms
```

Multiple measurements are collected and summarized using statistics such as:

```text
Mean
P50
P95
```

---

# 7. Mean Latency

Mean is the **average latency**.

Formula:

```text
Mean =
Sum of all latency measurements
───────────────────────────────
Number of measurements
```

Example:

```text
5 ms
7 ms
6 ms
8 ms
4 ms
```

Mean:

```text
(5 + 7 + 6 + 8 + 4) / 5

= 6 ms
```

Interpretation:

> On average, the request took approximately 6 ms.

---

# 8. P50 Latency

**P50 = 50th percentile = median latency.**

It represents the middle of the latency distribution.

Example:

```text
Sorted latency:

3 ms
4 ms
5 ms  ← P50
7 ms
12 ms
```

Therefore:

```text
P50 = 5 ms
```

Interpretation:

> Approximately 50% of requests completed in 5 ms or less.

P50 gives a good idea of the **typical request experience**.

---

# 9. P95 Latency

**P95 = 95th percentile latency.**

Interpretation:

> Approximately 95% of requests complete at or below the P95 latency, while the slowest ~5% take longer.

For example:

```text
P95 = 20 ms
```

means approximately:

```text
95% requests ≤ 20 ms
5% requests  > 20 ms
```

P95 is useful for understanding **tail latency** — occasional slow requests that the average can hide.

---

# 10. Why Mean Alone Is Not Enough

Consider two systems:

```text
                 Mean      P95

System A          8 ms      11 ms
System B          8 ms      35 ms
```

Their mean latency is identical.

But System B occasionally becomes significantly slower.

Therefore:

```text
Mean
→ Average performance

P50
→ Typical performance

P95
→ Slow-tail performance
```

This is why production systems often report percentile latency in addition to averages.

---

# 11. Latency Benchmarking in Our Milvus Assignment

We are comparing three Milvus indexes:

```text
FLAT
HNSW
IVF_FLAT
```

Our objective is:

> Compare the **search speed of the indexes themselves**.

Therefore, query embedding time should NOT be included in this particular benchmark.

---

# 12. Correct Benchmark Design

For every evaluation question:

```text
Question
   ↓
Generate Query Embedding ONCE
   ↓
Query Vector
   │
   ├────→ FLAT
   │       ↓
   │    Measure Search Latency
   │
   ├────→ HNSW
   │       ↓
   │    Measure Search Latency
   │
   └────→ IVF_FLAT
           ↓
        Measure Search Latency
```

The **same query vector** is searched against all three indexes.

This creates a fair comparison.

---

# 13. Why Exclude Embedding Time?

Suppose:

```text
Embedding API = 120 ms

FLAT search     = 9 ms
HNSW search     = 3 ms
IVF_FLAT search = 5 ms
```

If embedding latency is included:

```text
FLAT     ≈ 129 ms
HNSW     ≈ 123 ms
IVF_FLAT ≈ 125 ms
```

The large embedding latency hides much of the difference between index types.

But our research question is:

> Which vector index performs the search faster?

Therefore, we isolate:

```text
FLAT     = 9 ms
HNSW     = 3 ms
IVF_FLAT = 5 ms
```

This follows an important benchmarking principle:

> **When comparing components, isolate the component being tested and keep other variables constant.**

---

# 14. Measuring Search Latency in Python

A high-resolution timer such as `time.perf_counter()` can be used:

```python
import time

query_vector = embedding_model.embed_query(question)

start = time.perf_counter()

results = search_milvus(query_vector)

end = time.perf_counter()

latency_ms = (end - start) * 1000
```

Notice:

```python
query_vector = embedding_model.embed_query(question)
```

happens **before** the timer starts.

Therefore the benchmark measures:

```text
Milvus search only
```

not:

```text
Embedding + Milvus search
```

---

# 15. Why Benchmark Multiple Queries?

A single measurement can be affected by:

- CPU load
- Memory/cache state
- Network overhead
- Background processes
- OS scheduling
- Warm-up effects
- Random temporary delays

Therefore:

```text
One Query
→ Weak evidence

Many representative queries
→ More reliable benchmark
```

In our assignment, we use the same **20 evaluation questions** for each index.

```text
20 Questions × 3 Indexes
              ↓
60 Search Measurements
```

---

# 16. Our Benchmark Output

For each index we summarize:

```text
Index        Mean     P50      P95

FLAT         ...
HNSW         ...
IVF_FLAT     ...
```

This tells us the **speed side** of index performance.

But speed alone is not enough.

We must also evaluate retrieval quality:

```text
Latency
   +
Recall@K / Retrieval Accuracy
   ↓
Overall Index Comparison
```

An extremely fast index is not useful if it frequently fails to retrieve the correct document.

---

# Interview Answer

### Q: What is latency?

> Latency is the amount of time required for a system to process a request and return a response. In a RAG system, latency can be measured at different levels, such as query embedding latency, vector-search latency, reranking latency, LLM generation latency, or complete end-to-end latency.

### Q: How would you benchmark vector index latency?

> I would generate each query embedding once and use the same query vector across all indexes. I would measure only the time from submitting the vector-search request until the Top-K results are returned. I would repeat this across multiple representative queries and report statistics such as mean, P50, and P95 latency.

### Q: Why wouldn't you include query embedding time when comparing FLAT, HNSW and IVF_FLAT?

> Because embedding generation is common to all three indexes and is not the component being evaluated. Including it could dominate the measurement and hide differences in index-search performance. I would measure embedding or end-to-end retrieval latency separately if I wanted to evaluate overall application performance.

### Q: Why report P95 in addition to average latency?

> Average latency can hide occasional slow requests. P95 shows the tail behavior: approximately 95% of requests complete at or below that value, while roughly 5% take longer. It therefore provides a better picture of performance consistency and worst-case user experience.

# Reranking & MMR — RAG Interview/Revision Notes

## 1. Why Do We Need Reranking?

In a RAG system, the vector database first retrieves chunks based on **embedding similarity**.

```text
User Query
    ↓
Query Embedding
    ↓
Vector Database (Milvus)
    ↓
Top-K Similar Chunks
```

However, the initially retrieved chunks may have two problems:

* **Ranking problem:** The most useful/relevant chunk may not be ranked first.
* **Redundancy problem:** Multiple retrieved chunks may contain almost the same information.

Therefore, we can apply a **second-stage retrieval process** before sending context to the LLM.

---

## 2. What is Reranking?

> **Reranking is the process of taking an initially retrieved set of candidate documents/chunks and reordering them using a more sophisticated relevance-scoring method.**

Typical architecture:

```text
Large Document Collection
        ↓
Fast Initial Retriever
        ↓
Top-20 Candidates
        ↓
Reranker
        ↓
Better Ranked Top-5
        ↓
LLM
```

The first-stage retriever needs to search a large number of documents quickly.

The reranker only needs to process a small candidate set, so it can use a more computationally expensive but potentially more accurate relevance method.

### Example

Initial vector retrieval:

```text
Query: "What causes sea level rise?"

1. General sea-level information
2. Historical sea-level measurements
3. Thermal expansion causes sea-level rise
4. Glacier melting contributes to sea-level rise
5. Coastal flooding observations
```

After reranking:

```text
1. Thermal expansion causes sea-level rise
2. Glacier melting contributes to sea-level rise
3. General sea-level information
4. Coastal flooding observations
5. Historical sea-level measurements
```

The documents may remain the same, but their **ranking improves**.

---

## 3. Common Reranking Approaches

Reranking can be performed using approaches such as:

```text
BM25 / lexical scoring
Cross-Encoder models
LLM-based relevance scoring
Hybrid relevance scores
MMR-based selection
```

The general goal is:

> **Retrieve broadly and quickly first, then refine the smaller candidate set more carefully.**

---

## 4. What is MMR?

**MMR = Maximal Marginal Relevance**

> **MMR is a document-selection strategy that balances relevance to the user's query with diversity among the selected documents.**

It tries to answer two questions:

```text
1. Is this chunk relevant to the query?

                 +

2. Does this chunk provide new information
   compared with chunks already selected?
```

---

## 5. Why Do We Need MMR?

Normal vector similarity search can return repetitive results.

For example:

```text
Query:
"What causes sea level rise?"

Top-5 Vector Results:

1. Thermal expansion
2. Thermal expansion
3. Thermal expansion
4. Thermal expansion
5. Glacier melting
```

These results are relevant, but highly **redundant**.

The LLM receives the same information repeatedly, wasting context-window tokens.

MMR may instead select:

```text
1. Thermal expansion
2. Glacier melting
3. Greenland ice-sheet loss
4. Antarctic ice-sheet loss
5. Land-water storage changes
```

Now the retrieved context covers multiple relevant aspects.

---

## 6. How Does MMR Work?

Conceptually:

```text
MMR Score

      Relevance
          ↓
How similar is this chunk
to the user's query?

          MINUS

      Redundancy
          ↓
How similar is this chunk
to chunks already selected?
```

A common mathematical formulation is:

```math
MMR(D) =
\lambda Sim(D,Q)
-
(1-\lambda)
\max_{D_s \in S} Sim(D,D_s)
```

Where:

```text
Q   = User query

D   = Candidate document

S   = Documents already selected

Sim(D,Q)
    = Relevance of candidate to query

Sim(D,Dₛ)
    = Similarity between candidate and
      already-selected documents

λ
    = Relevance-vs-diversity control
```

---

## 7. Role of Lambda (λ)

`λ` controls the trade-off between:

```text
Relevance ↔ Diversity
```

### Higher λ

More importance is given to **query relevance**.

```text
Higher λ
   ↓
More relevance
   ↓
Potentially more redundancy
```

### Lower λ

More importance is given to **diversity**.

```text
Lower λ
   ↓
More diversity
   ↓
Potentially less direct relevance
```

Therefore:

```text
             λ
             │
      Controls Balance
             │
     ┌───────┴───────┐
     ↓               ↓
 Relevance        Diversity
```

---

## 8. How MMR Selects Documents

Suppose initial retrieval gives 20 candidate chunks.

MMR first selects a highly relevant chunk:

```text
Selected:
Thermal expansion
```

For the next chunk, it doesn't only ask:

> Which remaining chunk is most relevant?

It also asks:

> Which relevant chunk gives me information different from "thermal expansion"?

So it may select:

```text
Glacier melting
```

rather than another almost-identical thermal-expansion chunk.

This process continues until the desired number of documents has been selected.

---

## 9. Retrieval vs Reranking vs MMR

| Concept       | Main Question                                    | Purpose                       |
| ------------- | ------------------------------------------------ | ----------------------------- |
| **Retrieval** | Which documents might be relevant?               | Find candidates               |
| **Reranking** | Which candidates are most relevant?              | Improve ordering              |
| **MMR**       | Which candidates are relevant AND non-redundant? | Improve relevance + diversity |

Easy way to remember:

```text
Retriever
   ↓
FIND candidates

Reranker
   ↓
ORDER candidates better

MMR
   ↓
SELECT relevant + diverse candidates
```

---

## 10. Reranking in a RAG Pipeline

Without reranking:

```text
Query
  ↓
Embedding
  ↓
Milvus
  ↓
Top-5
  ↓
LLM
```

With second-stage refinement:

```text
Query
  ↓
Embedding
  ↓
Milvus
  ↓
Retrieve Top-20 Candidates
  ↓
Reranking / MMR
  ↓
Select Best Top-5
  ↓
Prompt Context
  ↓
LLM
  ↓
Final Answer
```

A common strategy is therefore:

> **Retrieve more candidates than you ultimately need, then refine/select a smaller high-quality set.**

For example:

```text
Milvus → Top 20
             ↓
         Reranker
             ↓
          Top 5
             ↓
            LLM
```

---

## 11. Why Reranking Helps RAG

Reranking can improve:

```text
Retrieval relevance
        ↓
Better context
        ↓
Less irrelevant information
        ↓
Better use of context window
        ↓
Potentially better LLM answers
```

MMR additionally helps by:

```text
Reducing duplicate chunks
        ↓
Increasing information diversity
        ↓
Reducing wasted context tokens
```

---

## 12. Important Interview Distinction

**Reranking is a broad concept.**

It means:

> Re-evaluating the initially retrieved candidates to improve their final ranking/selection.

**MMR is a specific selection strategy** that considers both relevance and redundancy/diversity.

Therefore:

```text
Reranking
   ↓
Broad second-stage concept

MMR
   ↓
Specific relevance + diversity strategy
```

MMR can be used during retrieval or as a post-retrieval selection step depending on the implementation.

---

## 13. Reranking vs Vector Search

Vector search needs to search a potentially huge corpus:

```text
100,000 chunks
      ↓
Vector Search
      ↓
Top 20
```

A sophisticated reranker doesn't need to examine all 100,000 documents.

It only processes:

```text
20 candidates
      ↓
Reranker
      ↓
5 best results
```

This creates a useful **speed-quality trade-off**:

```text
Fast Retriever
      +
Accurate/Selective Reranker
      ↓
Efficient + High-quality Retrieval
```

---

## 14. Interview Answer

If asked **"What is reranking in RAG?"**:

> **Reranking is a second-stage retrieval technique where an initial fast retriever returns a candidate set, such as the Top-20 documents, and a more sophisticated scoring method re-evaluates those candidates to select or reorder the most relevant documents before passing them to the LLM.**

If asked **"What is MMR?"**:

> **Maximal Marginal Relevance (MMR) is a selection strategy that balances query relevance with result diversity. It rewards documents that are relevant to the query while penalizing documents that are too similar to already-selected results, thereby reducing redundant context.**

### One-line revision

> **Retriever finds candidates → Reranker improves relevance/order → MMR balances relevance and diversity to reduce redundant context.**


what recall answers us tell in marathi in 1 line

Recall आपल्याला सांगतो की, जेवढे खरोखर relevant/correct documents होते, त्यापैकी किती documents आपल्या retrieval system ने successfully retrieve केले.

so high recall is better

Yes, higher recall is generally better. ✅

High Recall = जास्तीत जास्त खरोखर relevant documents successfully retrieve झाले.

उदा. 10 relevant chunks असतील:

Retrieved 9 → Recall = 90% ✅
Retrieved 5 → Recall = 50% ❌

तुमच्या FLAT vs HNSW vs IVF_FLAT comparison मध्ये ideally आपल्याला high Recall@K + low latency हवे आहे.

## What does `avg_pairwise_similarity` mean?

`avg_pairwise_similarity` is basically a **redundancy meter** for the chunks that end up in the final RAG context.

It measures:

> **How similar are the selected chunks to each other?**

It does **not** measure how similar the chunks are to the user query.

---

## How is it calculated?

The function is:

```python
def avg_pairwise_similarity(chunks: list[dict]) -> float:
    if len(chunks) < 2:
        return 0.0

    sims = [
        cosine_similarity(
            chunks[i]["embedding"],
            chunks[j]["embedding"]
        )
        for i in range(len(chunks))
        for j in range(i + 1, len(chunks))
    ]

    return sum(sims) / len(sims)
```

Suppose the final context contains **5 chunks**:

```text
Chunk 1
Chunk 2
Chunk 3
Chunk 4
Chunk 5
```

We compare every unique pair:

```text
Chunk 1 ↔ Chunk 2
Chunk 1 ↔ Chunk 3
Chunk 1 ↔ Chunk 4
Chunk 1 ↔ Chunk 5

Chunk 2 ↔ Chunk 3
Chunk 2 ↔ Chunk 4
Chunk 2 ↔ Chunk 5

Chunk 3 ↔ Chunk 4
Chunk 3 ↔ Chunk 5

Chunk 4 ↔ Chunk 5
```

For 5 chunks, there are:

```text
C(5,2) = 10 unique pairs
```

For each pair, the system calculates **cosine similarity between their embeddings**.

For example:

```text
Chunk 1 ↔ Chunk 2 = 0.82
Chunk 1 ↔ Chunk 3 = 0.76
Chunk 1 ↔ Chunk 4 = 0.61
...
Chunk 4 ↔ Chunk 5 = 0.65
```

Then all 10 similarity scores are averaged:

```text
Average Pairwise Similarity
=
Sum of similarities between all chunk pairs
÷
Number of pairs
```

That gives one diversity/redundancy score for that question's selected context.

---

## How to Interpret the Number

### High `avg_pairwise_similarity`

```text
Closer to 1.0
      ↓
Chunks are highly similar
      ↓
More redundant information
```

For example:

```text
Chunk 1 → Thermal expansion causes sea-level rise
Chunk 2 → Ocean warming causes thermal expansion
Chunk 3 → Thermal expansion increases sea levels
Chunk 4 → Warmer oceans expand and raise sea level
Chunk 5 → Thermal expansion is a major sea-level contributor
```

All five chunks may be relevant, but they mostly repeat the same information.

---

### Lower `avg_pairwise_similarity`

```text
Lower similarity
      ↓
Chunks are more different
      ↓
More diverse information
```

For example:

```text
Chunk 1 → Thermal expansion
Chunk 2 → Glacier melting
Chunk 3 → Greenland ice-sheet loss
Chunk 4 → Antarctic ice-sheet loss
Chunk 5 → Land-water storage changes
```

These chunks provide different pieces of information about the same broader question.

---

## Why It Matters in RAG

Suppose the LLM receives 5 highly similar chunks:

```text
Context window
│
├── Thermal expansion
├── Thermal expansion
├── Thermal expansion
├── Thermal expansion
└── Thermal expansion
```

You're technically giving the LLM 5 chunks, but you're wasting much of the context window because they repeat the same information.

Ideally, you want:

```text
Context window
│
├── Thermal expansion
├── Glacier melting
├── Greenland ice sheet
├── Antarctic ice sheet
└── Land-water storage
```

Now the LLM receives **more information coverage using the same number of chunks**.

---

## Why We Use This Metric for MMR

The whole purpose of MMR is to reduce redundancy.

Therefore, when comparing:

```text
Plain Top-5
vs.
MMR Top-5
```

we expect:

```text
avg_pairwise_similarity
        ↓
Lower after MMR
        ↓
Less redundancy
        ↓
More diverse context
```

For your results:

```text
Plain Top-5 → 0.702
MMR Top-5   → 0.641
```

So:

```text
0.702 → 0.641
        ↓
Pairwise similarity decreased
        ↓
Selected chunks became less similar
        ↓
MMR successfully increased diversity
```

## One-Line Revision

> **`avg_pairwise_similarity` measures how semantically similar the retrieved chunks are to each other; lower values mean less redundancy and more diverse context for the LLM.**


## What does `avg_distinct_chunk_types` mean?

`avg_distinct_chunk_types` measures **how many different chunk types — `text`, `table`, and `image` — appear together in the final selected chunks**.

It is specifically a measure of **modality diversity**.

---

## How is it calculated?

From the code:

```python
"distinct_chunk_types": len(
    set(c["chunk_type"] for c in selected)
)
```

Suppose the final top-5 contains:

```text
text
text
text
table
image
```

First:

```text
set(...) = {"text", "table", "image"}
```

Then:

```text
distinct_chunk_types = 3
```

So for one query:

* 5 `text` chunks → **1 distinct chunk type**
* 4 `text` + 1 `table` → **2 distinct chunk types**
* `text` + `table` + `image` all present → **3 distinct chunk types**

Since your corpus has only three chunk types, the possible range is:

```text
1 → 3
```

---

## What does `avg_distinct_chunk_types` mean?

You calculate `distinct_chunk_types` separately for each of the 20 questions, then average those values.

For example:

```text
Question 1 → 1 type
Question 2 → 2 types
Question 3 → 1 type
Question 4 → 3 types
...
```

The average across all questions becomes:

```text
avg_distinct_chunk_types
```

---

## Your Results

```text
Plain Top-5 → 1.35
MMR Top-5   → 1.45
```

This means most retrieved sets are still dominated by **one modality**, usually text, but MMR slightly increases the chance of including another modality such as a table or image.

```text
1.35 → 1.45
        ↓
More modality variety
        ↓
MMR is slightly increasing
text/table/image diversity
```

The effect is modest because your corpus itself is heavily text-dominated:

```text
Text  → 1205 chunks
Table →   25 chunks
Image →   71 chunks
-------------------
Total → 1301 chunks
```

So text naturally has many more opportunities to appear in the top results.

---

## Difference from `avg_pairwise_similarity`

These two diversity metrics measure different things.

| Metric                     | Measures                                                                      |
| -------------------------- | ----------------------------------------------------------------------------- |
| `avg_pairwise_similarity`  | **Semantic diversity** — how similar the selected chunks are to each other    |
| `avg_distinct_chunk_types` | **Modality diversity** — how many of `text`, `table`, `image` are represented |

For example, these five chunks:

```text
Text: thermal expansion
Text: glacier melting
Text: ice-sheet loss
Text: land-water storage
Text: ocean warming
```

could have **low pairwise similarity** because they discuss different topics, but:

```text
distinct_chunk_types = 1
```

because every chunk is still `text`.

By contrast:

```text
Text
Text
Table
Image
Text
```

gives:

```text
distinct_chunk_types = 3
```

because all three modalities are represented.

### One-line revision

> **`avg_distinct_chunk_types` measures how many different modalities (`text`, `table`, `image`) appear in the retrieved context on average; a higher value means greater multimodal diversity.**


# BM25 — RAG / Interview Revision Notes

## 1. What is BM25?

**BM25 (Best Matching 25)** is a traditional **lexical / keyword-based ranking algorithm** used in Information Retrieval.

> **BM25 ranks documents based on how well the important keywords in a query match the words present in each document.**

Easy distinction:

```text
BM25          → Keyword / lexical matching
Vector Search → Semantic / meaning-based matching
```

---

## 2. Simple Example

Query:

```text
"SSP5-8.5 sea level projection"
```

Documents:

```text
Document A:
"SSP5-8.5 projects significant sea level rise by 2100."

Document B:
"Under high-emission scenarios, ocean levels are
expected to increase substantially."
```

BM25 will likely strongly favor **Document A** because exact terms such as:

```text
SSP5-8.5
sea
level
projection/projects
```

occur in the document.

A vector search may also identify **Document B** because its **meaning** is similar even though the exact wording differs.

---

## 3. How Does BM25 Calculate Relevance?

BM25 mainly considers three concepts:

### A. Term Frequency (TF)

**How often does a query term occur in a document?**

```text
Query: "sea level"

Document A:
"Sea level is increasing..."

Document B:
"Sea level changes ... sea level rise ... sea level projections..."
```

More occurrences can increase the BM25 score.

However, BM25 uses **term-frequency saturation**.

This means:

> A keyword appearing 20 times doesn't automatically make the document 20× more relevant.

---

### B. Inverse Document Frequency (IDF)

BM25 gives more importance to **rare/informative terms** and less importance to extremely common terms.

For your IPCC corpus:

```text
"the"       → extremely common → low importance

"climate"   → common           → relatively less informative

"SSP5-8.5"  → relatively rare  → highly informative
```

Conceptually:

```text
Rare term
   ↓
Higher IDF
   ↓
More importance

Common term
   ↓
Lower IDF
   ↓
Less importance
```

---

### C. Document Length Normalization

Long documents naturally contain more words and therefore have more chances to match query keywords.

BM25 adjusts for document length so that:

> A very long chunk does not automatically receive a high score simply because it contains more words.

Therefore, conceptually:

```text
                 BM25 Score
                    │
        ┌───────────┼───────────┐
        ↓           ↓           ↓
       TF          IDF     Length Normalization
        ↓           ↓           ↓
 How often?    How rare?   How long is doc?
```

---

## 4. BM25 Formula

A common simplified BM25 formulation is:

```math
Score(D,Q)=\sum_{t \in Q} IDF(t)
\frac{TF(t,D)(k_1+1)}
{TF(t,D)+k_1(1-b+b\frac{|D|}{avgdl})}
```

Where:

* `Q` = Query
* `D` = Document/chunk
* `TF(t,D)` = frequency of term `t` in document
* `IDF(t)` = importance/rarity of term
* `|D|` = document length
* `avgdl` = average document length
* `k1` = controls term-frequency saturation
* `b` = controls document-length normalization

For interviews, understanding the **intuition** is usually more important than memorizing the complete formula.

---

## 5. BM25 vs Vector Search

This distinction is very important for RAG.

| BM25                                      | Vector Search                      |
| ----------------------------------------- | ---------------------------------- |
| Lexical search                            | Semantic search                    |
| Keyword-based                             | Meaning-based                      |
| Exact terms are important                 | Conceptual similarity is important |
| No dense embeddings required              | Requires embeddings                |
| Strong for IDs, names, codes, terminology | Strong for synonyms/paraphrases    |

Example:

```text
Query:
"ocean temperature increase"
```

Document:

```text
"Seawater is becoming warmer due to global warming."
```

There may be limited exact keyword overlap.

Vector search can understand:

```text
ocean ≈ seawater
temperature increase ≈ becoming warmer
```

BM25 does not inherently understand those semantic relationships the same way embeddings do.

---

## 6. Why is BM25 Useful in RAG?

Embedding retrieval is excellent for semantic meaning but can sometimes miss or under-rank **exact terminology**.

BM25 is particularly useful for:

```text
Technical terms
Model names
Product names
Abbreviations
Acronyms
IDs
Codes
Scientific terminology
Exact phrases
```

For your IPCC dataset, examples include:

```text
SSP5-8.5
SSP2-4.5
CMIP6
AR6
RCP8.5
Figure 9.27
```

Therefore BM25 and vector retrieval **complement each other**.

---

## 7. Hybrid Retrieval

A strong RAG architecture can use both:

```text
                       Query
                         │
              ┌──────────┴──────────┐
              ↓                     ↓
            BM25              Vector Search
              ↓                     ↓
       Lexical Results        Semantic Results
              │                     │
              └──────────┬──────────┘
                         ↓
                   Combine / Fuse
                         ↓
                    Hybrid Ranking
                         ↓
                       Top-K
```

This is called **Hybrid Retrieval/Search**.

> **Hybrid retrieval combines lexical search such as BM25 with semantic vector search.**

---

## 8. Why Hybrid Search?

Because each method compensates for the other's weaknesses.

```text
BM25
"What WORDS match?"
        +
Vector Search
"What MEANING matches?"
        ↓
Hybrid Retrieval
```

Example:

```text
Query:
"SSP5-8.5 ocean warming projection"
```

BM25 can strongly identify:

```text
SSP5-8.5
```

while vector search can identify semantically related passages discussing:

```text
future ocean warming under high-emission scenarios
```

Together, they can produce a stronger candidate set.

---

## 9. What is Hybrid Ranking?

After BM25 and vector search produce results:

```text
BM25 Results            Vector Results
     ↓                         ↓
     A                         C
     B                         A
     D                         E
     C                         B
      \                       /
       \                     /
        └────── Combine ─────┘
                 ↓
          Hybrid Ranking
                 ↓
         Final Ranked List
```

The scores/ranks from both retrieval systems need to be **combined/fused**.

Conceptually, you might imagine:

```math
HybridScore =
\alpha(BM25Score)
+
(1-\alpha)(VectorScore)
```

However, raw BM25 and vector scores are often on different scales, so in practice you generally need **normalization** or a rank-fusion technique such as **Reciprocal Rank Fusion (RRF)** rather than blindly adding raw scores.

---

## 10. Where Does MMR Fit?

After hybrid retrieval, MMR can optionally be used to select a **relevant but diverse** final context set.

```text
                         Query
                           │
              ┌────────────┴────────────┐
              ↓                         ↓
            BM25                  Vector Search
              │                         │
              └────────────┬────────────┘
                           ↓
                     Hybrid Ranking
                           ↓
                    Top-20 Candidates
                           ↓
                          MMR
                           ↓
                 Relevant + Diverse
                      Top-5 Chunks
                           ↓
                          LLM
```

Responsibilities:

```text
BM25
→ Keyword relevance

Vector Search
→ Semantic relevance

Hybrid Ranking
→ Combine both retrieval signals

MMR
→ Reduce redundancy and increase diversity

LLM
→ Generate final answer
```

---

## 11. BM25 vs MMR

Do **not** confuse these.

### BM25 asks:

> **Which documents contain important query terms?**

### MMR asks:

> **Which relevant documents should I select while avoiding redundant information?**

So:

```text
BM25 → Relevance through lexical matching

MMR  → Relevance + diversity
```

They can therefore be used **together** rather than being mutually exclusive.

---

## 12. Your Assignment Architecture

A good experimental setup for your multimodal RAG assignment is:

```text
                 User Query
                     ↓
           ┌─────────┴─────────┐
           ↓                   ↓
         BM25             Query Embedding
           ↓                   ↓
    Lexical Retrieval        Milvus
                               ↓
                         Vector Retrieval
           │                   │
           └─────────┬─────────┘
                     ↓
               Hybrid Ranking
                     ↓
              Top-N Candidates
                     ↓
                    MMR
                     ↓
               Final Top-K
                     ↓
              Prompt Template
                     ↓
                    LLM
                     ↓
               Final Answer
```

But for evaluation, don't only test the full pipeline. Compare the components separately:

```text
1. Vector Search
2. Vector Search + MMR
3. BM25 + Vector → Hybrid
4. BM25 + Vector → Hybrid → MMR
```

Then compare metrics such as:

```text
Recall@K
Precision@K
Latency
Diversity
Final RAG answer quality
```

This lets you determine **which technique actually contributed what**.

---

## 13. Key Interview Points

### BM25

> A lexical ranking algorithm based primarily on term frequency, inverse document frequency, and document-length normalization.

### Vector Search

> Uses embeddings to retrieve documents based on semantic similarity.

### Hybrid Retrieval

> Combines lexical retrieval such as BM25 with dense vector retrieval to capture both exact keyword matches and semantic similarity.

### MMR

> Selects documents by balancing query relevance and diversity, reducing redundant context.

---

## ⭐ Quick Revision

```text
BM25
│
├── Lexical / keyword search
├── Term Frequency (TF)
├── Inverse Document Frequency (IDF)
├── Document length normalization
├── Great for exact terms / IDs / terminology
└── Does NOT inherently understand semantics like embeddings


Vector Search
│
├── Embedding-based
├── Semantic search
└── Great for meaning / synonyms / paraphrases


BM25 + Vector
        ↓
Hybrid Retrieval / Ranking
        ↓
Optional MMR
        ↓
Relevant + Diverse Top-K
        ↓
LLM
```

**One-line interview answer:**

> **BM25 is a lexical ranking algorithm that uses term frequency, inverse document frequency, and document-length normalization to rank documents; in RAG, it is commonly combined with semantic vector search for hybrid retrieval.**


## Can We Use MMR Instead of RRF?

Not really — **RRF and MMR solve different problems**, so MMR is not a direct replacement for RRF in a hybrid BM25 + vector pipeline.

### Think of Their Jobs Separately

You have two ranked lists:

```text
BM25 Results                 Vector Results
(keyword-based)              (semantic)
     ↓                            ↓
A, B, C, D                  C, E, A, F
     \                            /
      \                          /
       └──────── RRF ───────────┘
                   ↓
          ONE combined ranking
                   ↓
             A, C, B, E...
```

**RRF (Reciprocal Rank Fusion)** answers:

> “I have rankings from BM25 and vector search. How do I **combine these rankings into one**?”

Now after RRF, you might still have redundant chunks:

```text
1. Thermal expansion
2. Thermal expansion
3. Thermal expansion
4. Glacier melting
5. Ice-sheet melting
...
```

That's where **MMR** comes in:

```text
RRF Top-20
    ↓
   MMR
    ↓
Final Top-5
```

MMR asks:

> “From these candidates, how do I select results that are **relevant but not repetitive**?”

---

## So for Your Assignment

A clean architecture would be:

```text
                   Query
                     │
          ┌──────────┴──────────┐
          ↓                     ↓
        BM25              Vector Search
          ↓                     ↓
    Ranked List A          Ranked List B
          │                     │
          └──────────┬──────────┘
                     ↓
                    RRF
                     ↓
             Hybrid Ranking
                     ↓
              Top-N Candidates
                     ↓
                    MMR
                     ↓
         Relevant + Diverse Top-K
                     ↓
                    LLM
```

So remember:

| Technique         | Job                                   |
| ----------------- | ------------------------------------- |
| **BM25**          | Keyword retrieval                     |
| **Vector Search** | Semantic retrieval                    |
| **RRF**           | **Fuse** BM25 + vector rankings       |
| **MMR**           | **Diversify/select** final candidates |

---

## Can You Technically Skip RRF and Use MMR?

You *could design* a custom procedure that merges the BM25 and vector candidate pools and then applies MMR.

However, **MMR by itself does not properly fuse BM25 ranking information with vector ranking information**.

MMR normally needs a relevance signal plus a redundancy/diversity signal:

```text
MMR
=
Query relevance
-
Redundancy with already-selected chunks
```

It does not inherently answer:

```text
BM25 says chunk A is rank #1
+
Vector search says chunk A is rank #3
+
BM25 says chunk B is rank #8
+
Vector search says chunk B is rank #1

→ How should these two rankings be combined?
```

That is exactly the problem **RRF** is designed to solve.

So if you simply merge the two candidate lists and run MMR, you may lose the explicit benefit of combining both:

* BM25 lexical ranking
* Vector semantic ranking

---

## Recommended Architecture

For the full hybrid pipeline:

```text
BM25
   +
Vector Search
   ↓
RRF
   ↓
Hybrid Top-N
   ↓
MMR
   ↓
Final Diverse Top-K
   ↓
LLM
```

So:

> **BM25 + Vector → RRF → MMR → Final Top-K**

This lets every component do one clear job.

---

## Good Experimental Setup

You can evaluate the stages separately:

```text
1. Vector Search
2. Vector Search + MMR
3. BM25 + Vector → RRF
4. BM25 + Vector → RRF → MMR
```

Then compare:

```text
Recall@K
Precision@K
Pairwise similarity
Distinct chunk types
Latency
Final answer quality
```

This tells you exactly what each technique contributes.

---

## Easy Memory Trick

> **RRF = Combine the rankings**

> **MMR = Remove redundancy while keeping relevance**

Or even shorter:

```text
RRF → Fusion

MMR → Diversity
```


# Stage 6 — Results Interpretation

The three retrieval strategies were evaluated on the same 20-question evaluation set using:

* **Recall@5** — whether the known-correct chunk appeared in the final top-5
* **Average pairwise similarity** — measures redundancy among the selected chunks; lower is better
* **Average distinct chunk types** — measures how many different chunk types (`text`, `table`, `image`) appear in the selected context; higher indicates greater modality diversity

## Results

| Method   | Recall@5 | Avg. Pairwise Similarity | Avg. Distinct Chunk Types |
| -------- | -------: | -----------------------: | ------------------------: |
| Baseline |     0.80 |                   0.7027 |                      1.35 |
| Hybrid   | **0.90** |                   0.6992 |                      1.30 |
| MMR      |     0.75 |               **0.6404** |                  **1.45** |

Since the evaluation set contains 20 questions:

```text
Baseline → 0.80 × 20 = 16/20 questions
Hybrid   → 0.90 × 20 = 18/20 questions
MMR      → 0.75 × 20 = 15/20 questions
```

---

## 1. Baseline Vector Retrieval

The baseline retrieves the top-5 chunks using only semantic vector similarity.

```text
Recall@5                  = 0.80
Avg. Pairwise Similarity  = 0.7027
Avg. Distinct Chunk Types = 1.35
```

This means the baseline successfully retrieved the known-correct chunk for **16 out of 20 questions**.

The baseline provides a reasonable balance between retrieval accuracy and diversity, but because it relies only on semantic similarity, it can miss chunks that contain important **exact keywords, technical terms, IDs, acronyms, or scientific terminology**.

---

## 2. Hybrid Retrieval — Best Recall

The hybrid pipeline combines:

```text
Vector Search Top-15
        +
BM25 Top-15
        ↓
       RRF
        ↓
Final Top-5
```

Its results were:

```text
Recall@5                  = 0.90
Avg. Pairwise Similarity  = 0.6992
Avg. Distinct Chunk Types = 1.30
```

Hybrid retrieval achieved the **highest Recall@5 of 0.90**, successfully retrieving the correct chunk for **18 out of 20 questions**.

Compared with the baseline:

```text
Baseline Recall@5 = 0.80
Hybrid Recall@5   = 0.90
```

This shows that BM25 and vector retrieval provide **complementary retrieval signals**.

Vector search captures:

```text
Semantic meaning
Synonyms
Paraphrases
Conceptual similarity
```

while BM25 captures:

```text
Exact keywords
Technical terminology
Model names
Acronyms
Codes
Numbers
Specific phrases
```

Therefore, BM25 can recover relevant chunks that semantic vector retrieval may under-rank.

### Key Interpretation

> **Hybrid retrieval improved evidence coverage by combining lexical and semantic retrieval.**

This makes Hybrid the strongest method when the main objective is retrieving the correct evidence.

---

## 3. MMR — Best Diversity

MMR retrieves a larger candidate pool and then selects the final top-5 by balancing:

```text
Query Relevance
      +
Diversity
      ↓
Final Top-5
```

Its results were:

```text
Recall@5                  = 0.75
Avg. Pairwise Similarity  = 0.6404
Avg. Distinct Chunk Types = 1.45
```

MMR achieved the **lowest Recall@5**, successfully retrieving the correct chunk for **15 out of 20 questions**.

However, it achieved the strongest diversity results.

---

## 4. Why Did MMR Reduce Recall?

MMR does not optimize only for retrieval relevance.

Its objective approximately balances:

```text
MMR Score
    =
Relevance to Query
    -
Redundancy with Already Selected Chunks
```

With:

```text
MMR_LAMBDA = 0.6
```

the system leans toward relevance while still applying a meaningful diversity penalty.

A highly relevant candidate may therefore be skipped if it is very similar to chunks already selected.

For example:

```text
Plain Vector Top-5

1. Chunk A
2. Chunk B
3. Chunk C
4. Chunk D
5. Ground-truth Chunk
```

MMR may decide that the ground-truth chunk is too similar to an already-selected result and replace it with a more novel candidate:

```text
MMR Top-5

1. Chunk A
2. Chunk B
3. Chunk C
4. Chunk D
5. Chunk F
```

This can reduce Recall@5 even though the resulting context becomes less repetitive.

Therefore:

> **MMR trades some pure retrieval relevance for greater diversity.**

---

## 5. Average Pairwise Similarity

The average pairwise similarity measures how similar the selected chunks are **to each other**.

It is a redundancy metric, not a query-relevance metric.

For five selected chunks, there are:

```text
C(5,2) = 10
```

unique chunk pairs.

The cosine similarity of each pair is calculated and then averaged.

Results:

```text
Baseline = 0.7027
Hybrid   = 0.6992
MMR      = 0.6404
```

Lower values indicate greater semantic diversity.

MMR reduced pairwise similarity from:

```text
0.7027 → 0.6404
```

compared with baseline.

This is approximately a **9% relative reduction in pairwise similarity**.

Therefore:

> **MMR successfully reduced semantic redundancy among the retrieved chunks.**

Hybrid's value:

```text
0.6992
```

is almost identical to baseline:

```text
0.7027
```

This is expected because RRF combines rankings but does not explicitly optimize diversity.

---

## 6. Average Distinct Chunk Types

The corpus contains three chunk types:

```text
text
table
image
```

The metric therefore ranges approximately from:

```text
1 → only one type appears
2 → two chunk types appear
3 → text + table + image all appear
```

Results:

```text
Baseline = 1.35
Hybrid   = 1.30
MMR      = 1.45
```

MMR achieved the highest modality diversity.

This means MMR was more likely to select a mixture such as:

```text
text
text
table
image
text
```

instead of:

```text
text
text
text
text
text
```

Therefore:

> **MMR improved both semantic diversity and chunk-type diversity.**

---

## 7. Why Did Hybrid Not Improve Diversity?

Hybrid retrieval combines BM25 and vector rankings using RRF.

RRF asks:

> Which chunks rank strongly across the lexical and semantic retrieval systems?

It does **not** ask:

> Are the selected chunks redundant with one another?

For example:

```text
BM25 Ranking

1. Text A
2. Text B
3. Table C

Vector Ranking

1. Text B
2. Text A
3. Image D
```

Both methods strongly rank `Text A` and `Text B`.

RRF therefore reinforces those chunks:

```text
Text A → strong in both rankings
Text B → strong in both rankings
```

This can improve retrieval relevance without increasing modality diversity.

Therefore:

```text
Hybrid → Better recall
MMR    → Better diversity
```

---

## 8. Main Trade-Off

The experiment demonstrates a clear trade-off between **retrieval accuracy** and **context diversity**.

| Objective                  | Best Method         |
| -------------------------- | ------------------- |
| Highest Recall@5           | **Hybrid**          |
| Correct evidence retrieval | **Hybrid**          |
| Exact keyword retrieval    | **Hybrid / BM25**   |
| Semantic retrieval         | **Vector / Hybrid** |
| Lowest redundancy          | **MMR**             |
| Semantic diversity         | **MMR**             |
| Chunk-type diversity       | **MMR**             |

### Overall Behavior

```text
Baseline
    ↓
Good semantic retrieval
Moderate recall
Moderate diversity


Hybrid
    ↓
Vector + BM25 + RRF
Highest recall
Best evidence coverage


MMR
    ↓
Relevance + diversity selection
Lower recall
Lowest redundancy
Highest diversity
```

---

## 9. Why Hybrid Is Selected for Stage 7

For Stage 7, the most important requirement is that the correct evidence reaches the LLM.

If the relevant chunk is not retrieved:

```text
Relevant evidence missing
        ↓
LLM does not receive the answer
        ↓
Generation quality may decrease
```

Hybrid retrieved the known-correct evidence for:

```text
18 / 20 questions
```

compared with:

```text
Baseline → 16 / 20
MMR      → 15 / 20
```

Therefore, Hybrid is selected as the final retrieval strategy for Stage 7.

The Stage 7 pipeline becomes:

```text
                    User Query
                        │
              ┌─────────┴─────────┐
              ↓                   ↓
         Vector Search           BM25
              ↓                   ↓
          Top-15               Top-15
              │                   │
              └─────────┬─────────┘
                        ↓
                       RRF
                        ↓
                 Hybrid Top-5
                        ↓
                 Prompt Template
                        ↓
                     GPT-4o
                        ↓
                  Final Answer
```

---

## 10. Optional Future Experiment — Hybrid + MMR

The current results suggest an additional possible experiment:

```text
Vector Search
      +
BM25
      ↓
RRF
      ↓
Hybrid Top-N
      ↓
MMR
      ↓
Final Diverse Top-K
```

The motivation is:

```text
Hybrid
→ Highest recall

MMR
→ Highest diversity
```

Combining them could potentially retain Hybrid's strong candidate coverage while using MMR to reduce redundancy in the final context.

However, this should be evaluated experimentally because applying MMR after hybrid retrieval could again remove a ground-truth chunk from the final top-5.

---

## Final Conclusion

> **Hybrid retrieval achieved the highest Recall@5 (0.90), improving over the vector-search baseline (0.80), demonstrating the complementary value of lexical BM25 and semantic vector retrieval. MMR reduced Recall@5 to 0.75 but substantially improved diversity, lowering average pairwise similarity from 0.703 to 0.640 and increasing the average number of distinct chunk types from 1.35 to 1.45. These results highlight a clear trade-off: hybrid ranking improves evidence coverage, whereas MMR primarily improves context diversity and reduces redundancy.**

### Final Method Selected for Stage 7

```text
Vector Search + BM25
        ↓
       RRF
        ↓
Hybrid Top-5
        ↓
Prompt + GPT-4o Generation
```

**Reason:** Hybrid achieved the highest retrieval Recall@5, and for answer generation, retrieving the correct supporting evidence is prioritized over maximizing diversity.


### RAGAS Article: https://www.vectara.com/blog/evaluating-rag

### RAG pipeline evaluate करताना मुख्यतः 4 components असतात:

- Question → User ने विचारलेला प्रश्न/query.
- Context → Retriever/Vector DB ने retrieve केलेले relevant chunks, जे LLM ला answer तयार करण्यासाठी दिले जातात.
- Answer → RAG system/LLM ने generate केलेला final response.
- Ground Truth Answer → आपण आधीपासून correct/expected answer म्हणून define केलेला reference answer, ज्याच्याशी generated answer compare करता येतो. Ground Truth = आपल्याला अपेक्षित असलेले correct/reference answer.

# RAGAS Metrics — सोप्या भाषेत

हे चार **RAGAS metrics** तुमचा RAG system वेगवेगळ्या angles ने किती चांगला perform करतो हे evaluate करतात.

---

## 1. Faithfulness — *“Answer context वर based आहे का?”*

LLM ने answer मध्ये केलेले claims **retrieved context मधूनच supported आहेत का** हे तपासते.

```text
Context:
Sea level rises due to thermal expansion.

Answer:
Sea level rises due to thermal expansion. ✅

Answer:
Sea level rises mainly due to earthquakes. ❌
```

दुसऱ्या answer मध्ये context support नसल्यामुळे **Faithfulness कमी**.

> **High Faithfulness = कमी hallucination / answer contextशी grounded आहे.**

यासाठी ground-truth answer आवश्यक नसतो.

---

## 2. Answer Relevancy — *“Answer ने actual question ला उत्तर दिलं का?”*

User ने जे विचारलं त्यालाच generated answer respond करतोय का हे तपासते.

```text
Question:
What causes sea level rise?

Answer:
Thermal expansion and melting land ice
contribute to sea level rise. ✅
```

पण answer फक्त general climate change बद्दल बोलत राहिला तर relevancy कमी होईल.

> **High Answer Relevancy = answer question ला directly address करतो.**

---

## 3. Context Precision — *“Retrieve केलेल्या chunks मध्ये useful chunks वर आहेत का?”*

Retriever ने आणलेल्या chunks मध्ये **relevant information किती focused आहे आणि relevant chunks ranking मध्ये वर आहेत का** हे पाहते.

उदा.:

```text
Retrieved Top-5:

1. Sea level rise causes       ✅
2. Thermal expansion           ✅
3. Glacier melting             ✅
4. Forest ecosystems           ❌
5. Agriculture                 ❌
```

हे relatively good.

पण:

```text
1. Agriculture                 ❌
2. Forest ecosystems           ❌
3. Sea level rise causes       ✅
4. Thermal expansion           ✅
5. Glacier melting             ✅
```

Relevant chunks आहेत, पण खाली आहेत → **Context Precision खराब होऊ शकते.**

> **High Context Precision = retrieved context मध्ये relevant information concentrated आहे आणि useful chunks appropriately high-ranked आहेत.**

---

## 4. Context Recall — *“Correct answer बनवण्यासाठी लागणारी माहिती retrieve झाली का?”*

हा metric विचारतो:

> **Ground-truth/correct answer तयार करण्यासाठी जी information आवश्यक आहे, ती retrieved context मध्ये sufficiently present आहे का?**

समजा correct answer ला तीन facts लागतात:

```text
Sea level rise causes:

1. Thermal expansion
2. Glacier melting
3. Ice-sheet loss
```

Retriever ने दिलं:

```text
Thermal expansion ✅
Glacier melting   ✅
Ice-sheet loss    ✅
```

→ **High Context Recall**

पण:

```text
Thermal expansion ✅
Glacier melting   ❌
Ice-sheet loss    ❌
```

→ **Lower Context Recall**

---

## चारही लक्षात ठेवण्याची सोपी trick

```text
Faithfulness
→ Answer CONTEXT वरूनच आला का?
→ Hallucination आहे का?

Answer Relevancy
→ Answer QUESTION ला योग्य उत्तर देतो का?

Context Precision
→ Retrieved relevant chunks योग्य/वरच्या positions ला आहेत का?

Context Recall
→ Correct answer साठी लागणारी INFORMATION retrieve झाली का?
```

आणखी सोपं:

> **Faithfulness = Answer ↔ Context**

> **Answer Relevancy = Answer ↔ Question**

> **Context Precision = Retrieved Context ची quality/ranking**

> **Context Recall = Retrieved Context ची completeness**
